In [ ]:
# %% [markdown]
# # Value-Targeted, Resource-Constrained RL Eavesdropper — Phase 1
# Simulator: hidden-demand traffic, BB84 key generation, capped pool,
# delayed key-pool ledger (dual-ledger design), detection statistic, POMDP env,
# and baselines for incremental availability damage under a stealth constraint.

%pip install -q \
    "stable-baselines3>=2.6,<3" \
    "sb3-contrib>=2.6,<3" \
    "gymnasium>=1.0,<2" \
    "tensorboard>=2.20,<3" \
    "tqdm>=4.66"
# %% Cell 1 — Imports & global config
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from collections import deque
from enum import IntEnum
from functools import lru_cache

# Run the complete notebook pipeline from start to finish.
SESSION_STAGE = "all"
# SB3 recommends CPU for PPO with MLP policies; benchmark locally before changing this.
PPO_DEVICE = "cpu"

RNG_GLOBAL_SEED = 12345


class Regime(IntEnum):
    CALM = 0
    ACTIVE = 1


class ValueClass(IntEnum):
    LOW = 0
    NORMAL = 1
    HIGH = 2


# Regime transition matrix (slow, sticky macro state — NOT observed by Eve directly)
REGIME_TRANS = np.array([
    [0.98, 0.02],   # from CALM
    [0.05, 0.95],   # from ACTIVE
])

# Value-class transition matrices, CONDITIONED on regime.
# Under ACTIVE regime, HIGH-value periods are far more likely and persistent.
VALUE_TRANS = {
    Regime.CALM: np.array([
        [0.85, 0.14, 0.01],   # from LOW
        [0.10, 0.85, 0.05],   # from NORMAL
        [0.05, 0.35, 0.60],   # from HIGH
    ]),
    Regime.ACTIVE: np.array([
        [0.60, 0.35, 0.05],
        [0.05, 0.70, 0.25],
        [0.02, 0.18, 0.80],
    ]),
}

JOINT_STATES = tuple((r, v) for r in Regime for v in ValueClass)
JOINT_TRANS = np.zeros((6, 6), dtype=float)
for i, (regime, value) in enumerate(JOINT_STATES):
    for j, (next_regime, next_value) in enumerate(JOINT_STATES):
        JOINT_TRANS[i, j] = REGIME_TRANS[regime, next_regime] * VALUE_TRANS[next_regime][value, next_value]


def stationary_distribution(transition, tol=1e-14, max_iter=100_000):
    distribution = np.full(transition.shape[0], 1.0 / transition.shape[0])
    for _ in range(max_iter):
        updated = distribution @ transition
        if np.max(np.abs(updated - distribution)) < tol:
            return updated / updated.sum()
        distribution = updated
    raise RuntimeError("stationary distribution did not converge")


STATIONARY_JOINT = stationary_distribution(JOINT_TRANS)
STATIONARY_VALUE = np.array([
    sum(STATIONARY_JOINT[i] for i, (_, value) in enumerate(JOINT_STATES) if value == v)
    for v in ValueClass
])


# Traffic generation parameters per value class (this is what Nexus-trace calibration
# should eventually replace/validate — for now these are synthetic placeholders).
CLASS_PARAMS = {
    ValueClass.LOW:    dict(rate=3.0,  burst_sigma=0.15, size_mean=250, size_std=60),
    ValueClass.NORMAL: dict(rate=10.0, burst_sigma=0.30, size_mean=450, size_std=120),
    ValueClass.HIGH:   dict(rate=25.0, burst_sigma=0.55, size_mean=900, size_std=250),
}


In [ ]:
# %% Cell 2 — Hidden-state process (Regime + ValueClass)
class HiddenStateProcess:
    """
    Two-timescale hidden state:
      - Regime: slow, sticky macro state (calm/active), NEVER observed by Eve.
      - ValueClass: faster value-context variable whose transition dynamics
        depend on Regime. This is C_t in the writeup — hidden from Eve,
        used only by the simulator for reward/eval.

    Rationale for the two-level design (see accompanying notes): if C_t were a
    simple single Markov chain, its autocorrelation decays geometrically with
    lag, so at large buffering delay Delta, current traffic tells you almost
    nothing about C_{t+Delta} and *no* policy (greedy or RL) could exploit it —
    the hypothesis "RL advantage grows with Delta" would have no mechanism to
    show up. The slow Regime variable provides a longer-memory, weakly
    observable signal (via persistent patterns in traffic statistics over a
    window) that a temporally-integrating policy can in principle extract even
    when C_t itself is uninformative at lag Delta. This is what "sequential
    decision-making" is actually supposed to buy Eve.
    """

    def __init__(self, rng: np.random.Generator):
        self.rng = rng
        self.reset()

    def reset(self):
        idx = int(self.rng.choice(len(JOINT_STATES), p=STATIONARY_JOINT))
        self.regime, self.value = JOINT_STATES[idx]
        return self.regime, self.value

    def step(self):
        # regime transition
        p_reg = REGIME_TRANS[self.regime]
        self.regime = Regime(self.rng.choice(2, p=p_reg))
        # value transition conditioned on (new) regime
        p_val = VALUE_TRANS[self.regime][self.value]
        self.value = ValueClass(self.rng.choice(3, p=p_val))
        return self.regime, self.value



In [ ]:
# %% Cell 3 — Traffic generator (raw process + noisy windowed observation)
class TrafficGenerator:
    """
    Given the current hidden ValueClass, generates raw per-round traffic
    (message count, total bytes), and maintains a rolling window to compute
    the *observable* metadata features Eve actually sees (with observation
    noise). This is the O_t <- C_t generative link.
    """

    def __init__(self, rng: np.random.Generator, window=5, metadata_noise_std=0.10):
        self.rng = rng
        self.window = window
        self.noise_std = metadata_noise_std
        self.count_hist = deque(maxlen=window)
        self.bytes_hist = deque(maxlen=window)

    def reset(self):
        self.count_hist.clear()
        self.bytes_hist.clear()
        # warm up window with zeros so early-episode features aren't NaN
        for _ in range(self.window):
            self.count_hist.append(0)
            self.bytes_hist.append(0.0)

    def step(self, value_class: ValueClass, shaper=None):
        p = CLASS_PARAMS[value_class]
        burst_mult = np.exp(self.rng.normal(0, p["burst_sigma"]))
        n_messages = self.rng.poisson(p["rate"] * burst_mult)

        if n_messages > 0:
            m, s = p["size_mean"], p["size_std"]
            sigma2 = np.log(1 + (s ** 2) / (m ** 2))
            mu = np.log(m) - sigma2 / 2
            sizes = self.rng.lognormal(mu, np.sqrt(sigma2), size=n_messages)
            total_bytes = float(sizes.sum())
        else:
            total_bytes = 0.0

        true_n_messages, true_total_bytes = n_messages, total_bytes
        obs_n_messages, obs_total_bytes = (
            shaper.shape(n_messages, total_bytes) if shaper is not None else (n_messages, total_bytes)
        )
        self.count_hist.append(obs_n_messages)
        self.bytes_hist.append(obs_total_bytes)

        counts = np.array(self.count_hist, dtype=float)
        byts = np.array(self.bytes_hist, dtype=float)

        traffic_rate_true = counts.mean()
        burstiness_true = counts.std() / (counts.mean() + 1e-6)   # coeff. of variation proxy
        mean_size_true = (byts.sum() / max(counts.sum(), 1e-6))

        def noisy(x):
            return max(0.0, x * (1 + self.rng.normal(0, self.noise_std)))

        obs = dict(
            traffic_rate_obs=noisy(traffic_rate_true),
            burstiness_obs=noisy(burstiness_true),
            size_obs=noisy(mean_size_true),
        )
        raw = dict(n_messages=true_n_messages, total_bytes=true_total_bytes,
                   observed_n_messages=obs_n_messages, observed_total_bytes=obs_total_bytes)
        return raw, obs



In [ ]:
# %% Cell 3b — Nexus trace ingestion
def load_nexus_trace(csv_path, round_seconds=1.0):
    """Expects CSV with columns: timestamp, conversation_id, size_bytes
    (export via Kafka consumer log of PUSH_MESSAGE/SEND_ACK events, or
    messaging-service metrics during a load-test run)."""
    df = pd.read_csv(csv_path)
    df["round_idx"] = (df["timestamp"] // round_seconds).astype(int)
    grouped = df.groupby("round_idx").agg(
        n_messages=("size_bytes", "count"), total_bytes=("size_bytes", "sum")
    ).reset_index()
    full_idx = pd.RangeIndex(grouped["round_idx"].min(), grouped["round_idx"].max() + 1)
    grouped = grouped.set_index("round_idx").reindex(full_idx, fill_value=0).reset_index()
    grouped.columns = ["round_idx", "n_messages", "total_bytes"]
    return grouped

# nexus_trace_df = load_nexus_trace("nexus_loadtest_trace.csv", round_seconds=1.0)


def load_collegemsg_trace(path, round_seconds=86400):
    """Load SNAP CollegeMsg (SRC DST UNIXTS) and aggregate messages by time bin.

    CollegeMsg contains timestamps and message edges, but no message sizes;
    total_bytes is therefore NaN and size-related calibration keeps its
    synthetic placeholder instead of inventing measured byte sizes.
    """
    raw = pd.read_csv(path, sep=r"\s+", names=["src", "dst", "timestamp"],
                      comment="#", compression="infer")
    raw["timestamp"] = pd.to_numeric(raw["timestamp"], errors="coerce")
    raw = raw.dropna(subset=["timestamp"])
    if raw.empty:
        raise ValueError("CollegeMsg file contains no valid timestamp rows")
    raw["round_idx"] = ((raw["timestamp"] - raw["timestamp"].min()) // round_seconds).astype(int)
    grouped = raw.groupby("round_idx").size().rename("n_messages")
    full_idx = pd.RangeIndex(0, int(grouped.index.max()) + 1)
    result = grouped.reindex(full_idx, fill_value=0).rename_axis("round_idx").reset_index()
    result["total_bytes"] = np.nan
    return result


In [ ]:
# %% Cell 3c — Calibrate CLASS_PARAMS against the empirical Nexus trace
def calibrate_class_params_from_trace(trace_df, window=5):
    rate = trace_df["n_messages"].rolling(window, min_periods=1).mean()
    q_low, q_high = rate.quantile([0.33, 0.66])
    buckets = {
        ValueClass.LOW: trace_df[rate <= q_low],
        ValueClass.NORMAL: trace_df[(rate > q_low) & (rate <= q_high)],
        ValueClass.HIGH: trace_df[rate > q_high],
    }
    calibrated = {}
    for vc, sub in buckets.items():
        if len(sub) < 5:
            calibrated[vc] = CLASS_PARAMS[vc]; continue
        msgs = sub["n_messages"].values
        params = dict(
            rate=float(msgs.mean()),
            burst_sigma=float(np.std(np.log(msgs.clip(min=1) / max(msgs.mean(), 1e-6) + 1e-6))),
        )
        sizes = (sub["total_bytes"] / sub["n_messages"].clip(lower=1)).to_numpy(dtype=float)
        sizes = sizes[np.isfinite(sizes)]
        if len(sizes):
            params.update(size_mean=float(sizes.mean()), size_std=float(sizes.std() + 1e-6))
        else:
            params.update(size_mean=CLASS_PARAMS[vc]["size_mean"],
                          size_std=CLASS_PARAMS[vc]["size_std"])
        calibrated[vc] = params
    return calibrated

def validate_calibration(trace_df, calibrated_params, n_rounds=None, seed=0,
                         output_path="nexus_calibration_validation.png"):
    n_rounds = n_rounds or len(trace_df)
    rng = np.random.default_rng(seed)
    gen = TrafficGenerator(rng, window=5, metadata_noise_std=0.0); gen.reset()
    global CLASS_PARAMS
    old_params = CLASS_PARAMS.copy(); CLASS_PARAMS.update(calibrated_params)
    synth_counts, hidden = [], HiddenStateProcess(rng)
    for _ in range(n_rounds):
        _, vc = hidden.step()
        raw, _ = gen.step(vc)
        synth_counts.append(raw["n_messages"])
    CLASS_PARAMS.update(old_params)

    def _acf(values, nlags=20):
        values = np.asarray(values, dtype=float)
        values = values - values.mean()
        denom = float(values @ values)
        return np.asarray([1.0] + [float(values[:-lag] @ values[lag:]) / denom
                                  if denom > 0 else 0.0 for lag in range(1, min(nlags, len(values)-1)+1)])

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].hist(trace_df["n_messages"], bins=30, alpha=0.5, density=True, label="real trace")
    ax[0].hist(synth_counts, bins=30, alpha=0.5, density=True, label="synthetic (calibrated)")
    ax[0].legend(); ax[0].set_title("messages/round distribution")
    ax[1].plot(_acf(trace_df["n_messages"], nlags=20), label="real trace")
    ax[1].plot(_acf(np.array(synth_counts), nlags=20), label="synthetic calibrated")
    ax[1].legend(); ax[1].set_title("autocorrelation")
    plt.tight_layout(); plt.savefig(output_path, dpi=120); plt.show()

# calibrated = calibrate_class_params_from_trace(nexus_trace_df)
# validate_calibration(nexus_trace_df, calibrated)
# CLASS_PARAMS.update(calibrated)   # only after validating the overlay plots


In [ ]:
# %% Cell 3d — Traffic shaping defenses
class TrafficShaper:
    """Applied only to what Eve observes; true consumption/demand is untouched."""
    def __init__(self, mode="none", rng=None, cover_traffic_rate=5.0, jitter_window=3):
        self.mode = mode
        self.rng = rng or np.random.default_rng(0)
        self.cover_traffic_rate = cover_traffic_rate
        self._jitter_buffer = deque(maxlen=jitter_window)

    def shape(self, n_messages, total_bytes):
        if self.mode == "none":
            return n_messages, total_bytes
        if self.mode == "constant_rate":
            target = max(60, int(self.cover_traffic_rate * 2))
            return target, target * 500.0
        if self.mode == "padding":
            target = max(60, int(self.cover_traffic_rate * 2))
            padded_count = max(int(n_messages), target)
            return padded_count, padded_count * 500.0
        if self.mode == "cover_traffic":
            cover = self.rng.poisson(self.cover_traffic_rate)
            return n_messages + cover, total_bytes + cover * 500.0
        if self.mode == "jitter":
            self._jitter_buffer.append((n_messages, total_bytes))
            return (np.mean([x[0] for x in self._jitter_buffer]),
                    np.mean([x[1] for x in self._jitter_buffer]))
        raise ValueError(f"unknown shaping mode {self.mode}")


In [ ]:
# %% Optional real-trace calibration: CollegeMsg (SNAP)
from pathlib import Path

COLLEGEMSG_PATH = "/kaggle/input/collegemsg/CollegeMsg.txt.gz"  # change to the attached Kaggle dataset path
RUN_COLLEGEMSG_VALIDATION = True
if RUN_COLLEGEMSG_VALIDATION:
    trace_path = Path(COLLEGEMSG_PATH)
    if trace_path.is_file():
        collegemsg_df = load_collegemsg_trace(trace_path, round_seconds=86400)
        calibrated_params = calibrate_class_params_from_trace(collegemsg_df, window=5)
        CLASS_PARAMS.update(calibrated_params)
        print(f"CollegeMsg: {len(collegemsg_df)} daily bins, {int(collegemsg_df['n_messages'].sum())} messages")
        print("Calibrated synthetic message-count dynamics from the real trace; CollegeMsg has no message sizes.")
        validate_calibration(collegemsg_df, calibrated_params, n_rounds=len(collegemsg_df), seed=2026,
                             output_path="/kaggle/working/collegemsg_calibration_validation.png")
    else:
        print(f"CollegeMsg not found at {trace_path}; attach SNAP CollegeMsg.txt.gz in Kaggle and update COLLEGEMSG_PATH.")



In [ ]:
# %% Cell 4 — BB84 intercept-resend physics
def h2(p):
    """Binary entropy, safe at p=0/1."""
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)


def secure_key_rate(qber):
    """
    Simplified asymptotic BB84 secure-key-rate approximation r(QBER) = 1 - 2*H2(QBER),
    clipped at 0 (hits 0 near ~11% QBER, matching the standard BB84 abort threshold).
    NOTE: this is a simplification for simulation purposes, not a finite-key
    composable-security proof. We use it only to make "how much *legitimate*
    usable key is generated this round" respond realistically to QBER.
    """
    r = 1 - 2 * h2(qber)
    return max(0.0, r)


# Standard individual intercept-resend attack result:
# Eve measures each intercepted qubit in a random basis and resends.
#   - matches Alice's basis w.p. 1/2 -> she learns the bit exactly, no error induced
#   - mismatches w.p. 1/2 -> she guesses w.p. 1/2 correct, and Bob sees an error w.p. 1/2
# => per intercepted+sifted bit: P(bit flip) = 0.25 ; P(Eve correct) = 0.75
QBER_SLOPE = 0.25
EVE_GUESS_ACCURACY = 0.75
EVE_INFO_PER_BIT = 1 + EVE_GUESS_ACCURACY * np.log2(EVE_GUESS_ACCURACY) + \
    (1 - EVE_GUESS_ACCURACY) * np.log2(1 - EVE_GUESS_ACCURACY)  # Shannon info, ~0.1887 bits

print(f"[sanity] Eve Shannon information per intercepted sifted bit: {EVE_INFO_PER_BIT:.4f} bits "
      f"(expected ~0.1887)")



In [ ]:
# %% Cell 5 — Dual-ledger key pool with explicit release delay (Delta)
#
# IMPORTANT DESIGN NOTE (this is the crux of the whole simulator — read carefully):
#
# We track TWO separate ledgers:
#   (a) `available_bits` — the legitimate, properly-secured key pool used for
#       encryption / key-exhaustion / latency (application-level reality).
#   (b) an "exposure" tag attached to each generated batch, representing the
#       AVERAGE fraction of pre-privacy-amplification information Eve holds
#       about that batch (proportional to the intercepted fraction f_t and her
#       per-bit information rate). This tag is tracked ALONGSIDE the secure
#       pool purely as an evaluation/scoring ledger — it does NOT reduce the
#       legitimate key pool's availability, and it is explicitly NOT
#       interpreted as residual information about the final privacy-amplified
#       key (which, if PA is done correctly, should be ~0 regardless of raw
#       interception). It is an "operational exposure" proxy: how much of the
#       raw material behind whatever gets consumed was, on average, partially
#       compromised pre-PA, weighted by how valuable that consumption turned
#       out to be.
#
# Each batch is release-delayed by Delta rounds after generation before it
# enters the "available for consumption" queue. Consumption draws FIFO. This
# delay is the operational analogue of QKD post-processing latency (sifting
# batch collection + error correction + privacy amplification take real time
# before key becomes usable) — so Delta has a genuine physical justification,
# not just being an artificial knob.
class KeyBatch:
    __slots__ = ("bits_remaining", "exposure_rate", "gen_round", "release_round")

    def __init__(self, bits, exposure_rate, gen_round, release_round):
        self.bits_remaining = bits
        self.exposure_rate = exposure_rate
        self.gen_round = gen_round
        self.release_round = release_round


class KeyPoolLedger:
    def __init__(self, delay_rounds: int, capacity_bits=None):
        self.delay = int(delay_rounds)
        self.capacity_bits = capacity_bits
        self.pipeline = deque()
        self.available = deque()
        self.total_generated = 0.0
        self.total_consumed = 0.0
        self.total_exposure_weighted_consumed = 0.0
        self.total_exhaustion_deficit = 0.0
        self.total_overflow = 0.0

    def add_batch(self, bits, exposure_rate, current_round):
        bits = float(bits)
        if bits <= 0:
            return
        self.pipeline.append(KeyBatch(bits, exposure_rate, current_round, current_round + self.delay))
        self.total_generated += bits

    def _advance_pipeline(self, current_round):
        while self.pipeline and self.pipeline[0].release_round <= current_round:
            batch = self.pipeline.popleft()
            room = (float('inf') if self.capacity_bits is None else
                    max(0.0, self.capacity_bits - self.pool_level()))
            accepted = min(batch.bits_remaining, room)
            self.total_overflow += batch.bits_remaining - accepted
            if accepted > 1e-9:
                self.available.append(KeyBatch(accepted, batch.exposure_rate,
                                               batch.gen_round, batch.release_round))

    def consume(self, demand_bits, current_round):
        """Return consumed bits, exposure proxy, and unmet demand."""
        self._advance_pipeline(current_round)
        bits_consumed = exposure_weighted = 0.0
        remaining_demand = float(demand_bits)
        while remaining_demand > 1e-9 and self.available:
            batch = self.available[0]
            take = min(batch.bits_remaining, remaining_demand)
            bits_consumed += take
            exposure_weighted += take * batch.exposure_rate
            batch.bits_remaining -= take
            remaining_demand -= take
            if batch.bits_remaining <= 1e-9:
                self.available.popleft()
        deficit = max(0.0, remaining_demand)
        self.total_consumed += bits_consumed
        self.total_exposure_weighted_consumed += exposure_weighted
        self.total_exhaustion_deficit += deficit
        return bits_consumed, exposure_weighted, deficit

    def pool_level(self):
        """Usable released key only; delayed in-transit batches are not spendable."""
        return sum(b.bits_remaining for b in self.available)

    def pipeline_level(self):
        """Key awaiting release; tracked separately and excluded from pool capacity."""
        return sum(b.bits_remaining for b in self.pipeline)

    def inventory_level(self):
        return self.pool_level() + self.pipeline_level()




In [ ]:
# %% Cell 6 (REPLACED) — Detection monitor (windowed CUSUM on public QBER samples)
class CusumDetector:
    """
    One-sided CUSUM on the publicly-announced QBER-estimation samples.
    S_t = max(0, S_{t-1} + (qber_hat_t - baseline - slack))
    Flags "detected" when S_t crosses a threshold h.
    """

    def __init__(self, baseline, slack, threshold):
        self.baseline = baseline
        self.slack = slack
        self.threshold = threshold
        self.s = 0.0
        self.max_s = 0.0
        self.triggered_round = None

    def update(self, qber_hat, round_idx):
        self.s = max(0.0, self.s + (qber_hat - self.baseline - self.slack))
        self.max_s = max(self.max_s, self.s)
        if self.s >= self.threshold and self.triggered_round is None:
            self.triggered_round = round_idx
        return self.s


@lru_cache(maxsize=32)
def calibrate_cusum_cached(qber_baseline, sample_size, n_rounds, slack,
                           target_false_alarm=0.01, n_calib_episodes=20000, seed=999):
    """Cached vectorized null calibration for episode-wise false-alarm control."""
    rng = np.random.default_rng(seed)
    stat = np.zeros(n_calib_episodes, dtype=float)
    peaks = np.zeros(n_calib_episodes, dtype=float)
    for _ in range(n_rounds):
        qhat = rng.binomial(sample_size, qber_baseline, size=n_calib_episodes) / sample_size
        stat = np.maximum(0.0, stat + qhat - qber_baseline - slack)
        peaks = np.maximum(peaks, stat)
    return float(np.quantile(peaks, 1.0 - target_false_alarm, method="higher"))


def calibrate_cusum_threshold(qber_baseline, sample_size, n_rounds, slack,
                               target_false_alarm=0.01, n_calib_episodes=20000, seed=999,
                               return_diagnostics=False):
    rng = np.random.default_rng(seed)
    stat = np.zeros(n_calib_episodes, dtype=float)
    peaks = np.zeros(n_calib_episodes, dtype=float)
    for _ in range(n_rounds):
        qhat = rng.binomial(sample_size, qber_baseline, size=n_calib_episodes) / sample_size
        stat = np.maximum(0.0, stat + qhat - qber_baseline - slack)
        peaks = np.maximum(peaks, stat)
    threshold = float(np.quantile(peaks, 1.0 - target_false_alarm, method="higher"))
    if return_diagnostics:
        return threshold, dict(mean=float(peaks.mean()), std=float(peaks.std()),
            frac_exactly_zero=float(np.mean(peaks == 0)), p50=float(np.quantile(peaks, .5)),
            p90=float(np.quantile(peaks, .9)), p99=float(np.quantile(peaks, .99)))
    return threshold


In [ ]:
# %% Cell 6b — Rolling pooled estimator for the HARD ABORT decision
class PooledAbortEstimator:
    """
    Real QKD protocols decide to abort based on accumulated statistics over a
    block, not a single small noisy sample. We pool raw (errors, samples)
    counts over the last `window` rounds and check the pooled ratio against
    qber_abort. This still responds fast to a SUSTAINED attack, but is much
    less prone to a single unlucky round of noise (or a single round of
    genuinely aggressive interception) triggering an instant, uninformative
    termination.
    """
    def __init__(self, window=3):
        self.window = window
        self.hist = deque(maxlen=window)  # list of (n_errors, n_samples)

    def update(self, n_errors, n_samples):
        self.hist.append((n_errors, n_samples))
        total_err = sum(e for e, _ in self.hist)
        total_n = sum(n for _, n in self.hist)
        pooled_qber = total_err / total_n if total_n > 0 else 0.0
        return pooled_qber


In [ ]:
# %% Cell 7 ? Config
@dataclass
class EnvConfig:
    n_qubits_per_round: int = 1000
    max_rounds: int = 300
    qber_baseline: float = 0.02
    qber_abort: float = 0.11
    sample_size_qber: int = 40                 # 8% of the expected 500 sifted bits
    intercept_levels: tuple = (0.0, 0.005, 0.01, 0.015, 0.02, 0.03, 0.04)
    total_budget_qubits: float = 3000.0         # constrains average attack near f=0.01
    key_bits_per_message: float | None = None  # calibrated below to the requested load
    load: float = 0.95                          # E[demand] / E[no-attack supply]
    pool_capacity_bits: float = 10000.0
    warm_start_rounds: float = 5.0
    damage_scale: float = 100.0
    delay_rounds: int = 0
    abort_penalty: float = 150.0
    window: int = 5
    metadata_noise_std: float = 0.10
    frame_stack: int = 5
    obs_mode: str = "full"
    shaping_mode: str = "none"
    rho: float = 1.0
    cusum_slack: float = 0.005
    target_false_alarm: float = 0.01
    cusum_threshold: float | None = None
    abort_window: int = 3
    detection_limit: float = 0.05
    seed: int = 0

    def __post_init__(self):
        if self.key_bits_per_message is None:
            expected_messages = sum(STATIONARY_VALUE[int(v)] * CLASS_PARAMS[v]["rate"] for v in ValueClass)
            expected_sifted = self.n_qubits_per_round * 0.5
            expected_secure = max(0.0, expected_sifted - self.sample_size_qber) * secure_key_rate(self.qber_baseline)
            self.key_bits_per_message = self.load * expected_secure / max(expected_messages, 1e-12)
        if not 0.0 <= self.rho <= 1.0:
            raise ValueError("rho must lie in [0, 1]")
        if self.pool_capacity_bits <= 0 or self.damage_scale <= 0:
            raise ValueError("pool_capacity_bits and damage_scale must be positive")


def baseline_secure_bits_per_round(cfg, qber=None):
    qber = cfg.qber_baseline if qber is None else qber
    n_sifted = cfg.n_qubits_per_round * 0.5
    remaining = max(0, n_sifted - cfg.sample_size_qber)
    return remaining * secure_key_rate(qber)


In [ ]:
# %% Cell 8 ? Eve environment: availability damage, independent streams, and risk latch
FEATURE_NAMES_FULL = ["qber_hat", "pool_level_norm", "traffic_rate_obs",
                      "burstiness_obs", "size_obs", "budget_frac"]
FEATURE_NAMES_QBER_POOL = ["qber_hat", "pool_level_norm", "budget_frac"]
FEATURE_NAMES_QBER_BUDGET = ["qber_hat", "budget_frac"]
FEATURE_NAMES_METADATA_BUDGET = ["qber_hat", "traffic_rate_obs", "burstiness_obs", "size_obs", "budget_frac"]
FEATURE_NAMES_DETECTOR = ["qber_hat", "cusum_frac", "budget_frac"]


class EveEnv:
    def __init__(self, cfg: EnvConfig):
        self.cfg = cfg
        self._master = np.random.default_rng(cfg.seed)
        self.pool_scale = cfg.pool_capacity_bits
        self.lmbda = 1.0
        if cfg.obs_mode == "full":
            self.feature_names = FEATURE_NAMES_FULL
        elif cfg.obs_mode == "qber_pool":
            self.feature_names = FEATURE_NAMES_QBER_POOL
        elif cfg.obs_mode == "metadata_budget":
            self.feature_names = FEATURE_NAMES_METADATA_BUDGET
        elif cfg.obs_mode in {"qber_budget", "qber_only_budget"}:
            self.feature_names = FEATURE_NAMES_QBER_BUDGET
        elif cfg.obs_mode == "detector_oracle":
            self.feature_names = FEATURE_NAMES_DETECTOR
        else:
            raise ValueError(f"unknown obs_mode: {cfg.obs_mode}")
        self.n_features = len(self.feature_names)
        self.obs_dim = self.n_features * cfg.frame_stack
        self.n_actions = len(cfg.intercept_levels)
        if cfg.cusum_threshold is None:
            cfg.cusum_threshold = calibrate_cusum_cached(
                cfg.qber_baseline, cfg.sample_size_qber, cfg.max_rounds,
                cfg.cusum_slack, cfg.target_false_alarm)
        self._init_rngs(cfg.seed)

    def _init_rngs(self, seed):
        streams = np.random.SeedSequence(int(seed)).spawn(6)
        (self.rng_hidden, self.rng_demand, self.rng_traffic,
         self.rng_phys, self.rng_shape, self.rng_coupling) = [np.random.default_rng(s) for s in streams]
        self.hidden = HiddenStateProcess(self.rng_hidden)
        self.traffic_demand = TrafficGenerator(self.rng_demand, window=self.cfg.window,
                                                metadata_noise_std=0.0)
        self.traffic = TrafficGenerator(self.rng_traffic, window=self.cfg.window,
                                        metadata_noise_std=self.cfg.metadata_noise_std)
        self.shaper = TrafficShaper(mode=self.cfg.shaping_mode, rng=self.rng_shape)

    def reset(self, seed=None):
        if seed is None:
            seed = int(self._master.integers(0, 2**31 - 1))
        self._init_rngs(seed)
        self.traffic.reset(); self.traffic_demand.reset()
        self.ledger = KeyPoolLedger(self.cfg.delay_rounds, self.cfg.pool_capacity_bits)
        self.shadow = KeyPoolLedger(self.cfg.delay_rounds, self.cfg.pool_capacity_bits)
        supply0 = baseline_secure_bits_per_round(self.cfg)
        warm_bits = self.cfg.warm_start_rounds * supply0
        self.ledger.add_batch(warm_bits, 0.0, current_round=-self.cfg.delay_rounds)
        self.shadow.add_batch(warm_bits, 0.0, current_round=-self.cfg.delay_rounds)
        self.ledger._advance_pipeline(0); self.shadow._advance_pipeline(0)
        self.detector = CusumDetector(self.cfg.qber_baseline, self.cfg.cusum_slack,
                                      self.cfg.cusum_threshold)
        self.abort_estimator = PooledAbortEstimator(window=self.cfg.abort_window)
        self.round = 0
        self.budget_remaining = self.cfg.total_budget_qubits
        self.done = self.hard_abort = self.detected = self.cusum_terminated = False
        self.termination_reason = None
        self.frame_buf = deque(maxlen=self.cfg.frame_stack)
        self._log = []
        zero_feat = np.zeros(self.n_features)
        for _ in range(self.cfg.frame_stack): self.frame_buf.append(zero_feat)
        return self._stacked_obs()

    def _make_feature_vec(self, qber_hat, traffic_obs, cusum_stat):
        budget_frac = self.budget_remaining / max(self.cfg.total_budget_qubits, 1e-12)
        if self.cfg.obs_mode == "full":
            return np.array([qber_hat, self.ledger.pool_level()/self.pool_scale,
                traffic_obs["traffic_rate_obs"], traffic_obs["burstiness_obs"],
                traffic_obs["size_obs"], budget_frac])
        if self.cfg.obs_mode == "qber_pool":
            return np.array([qber_hat, self.ledger.pool_level()/self.pool_scale, budget_frac])
        if self.cfg.obs_mode == "metadata_budget":
            return np.array([qber_hat, traffic_obs["traffic_rate_obs"],
                traffic_obs["burstiness_obs"], traffic_obs["size_obs"], budget_frac])
        if self.cfg.obs_mode in {"qber_budget", "qber_only_budget"}:
            return np.array([qber_hat, budget_frac])
        return np.array([qber_hat, cusum_stat/max(self.cfg.cusum_threshold,1e-12), budget_frac])

    def _stacked_obs(self):
        return np.concatenate(list(self.frame_buf)).astype(np.float32)

    def step(self, action_idx):
        assert not self.done, "step() called after episode termination"
        self.round += 1
        regime, value_class = self.hidden.regime, self.hidden.value
        raw_demand, _ = self.traffic_demand.step(value_class)
        obs_class = (value_class if self.rng_coupling.random() < self.cfg.rho else
                     ValueClass(self.rng_coupling.choice(3, p=STATIONARY_VALUE)))
        raw_metadata, traffic_obs = self.traffic.step(obs_class, shaper=self.shaper)
        demand_bits = raw_demand["n_messages"] * self.cfg.key_bits_per_message
        metadata_overhead = max(0.0, raw_metadata["observed_total_bytes"] - raw_metadata["total_bytes"]) / max(raw_metadata["total_bytes"], 1.0)
        metadata_throughput_loss = 0.0  # shaping changes Eve's view, not application delivery
        metadata_latency = ((self.cfg.window - 1) / 2.0 if self.cfg.shaping_mode == "jitter" else 0.0)

        f_requested = self.cfg.intercept_levels[int(action_idx)]
        if self.budget_remaining <= 1e-9:
            f_requested = 0.0
        requested_qubits = f_requested * self.cfg.n_qubits_per_round
        actual_qubits = min(requested_qubits, self.budget_remaining)
        f_actual = actual_qubits / self.cfg.n_qubits_per_round
        self.budget_remaining = max(0.0, self.budget_remaining - actual_qubits)

        n_sifted = self.rng_phys.binomial(self.cfg.n_qubits_per_round, 0.5)
        true_qber = min(1.0, self.cfg.qber_baseline + f_actual * QBER_SLOPE)
        n_sample = min(self.cfg.sample_size_qber, n_sifted)
        n_errors = self.rng_phys.binomial(n_sample, true_qber) if n_sample else 0
        qber_hat = n_errors / n_sample if n_sample else self.cfg.qber_baseline
        pooled_qber = self.abort_estimator.update(n_errors, n_sample)
        hard_abort = pooled_qber >= self.cfg.qber_abort
        remaining_for_key = max(0, n_sifted - n_sample)
        rate = secure_key_rate(true_qber) if not hard_abort else 0.0
        secure_bits_generated = remaining_for_key * rate
        self.ledger.add_batch(secure_bits_generated, f_actual * EVE_INFO_PER_BIT, self.round)
        self.shadow.add_batch(remaining_for_key * secure_key_rate(self.cfg.qber_baseline), 0.0, self.round)
        _, exposure_weighted, deficit = self.ledger.consume(demand_bits, self.round)
        _, _, shadow_deficit = self.shadow.consume(demand_bits, self.round)
        damage = deficit - shadow_deficit
        reward = damage / self.cfg.damage_scale

        was_detected = self.detected
        cusum_stat = self.detector.update(qber_hat, self.round)
        cusum_newly_triggered = (not was_detected and self.detector.triggered_round is not None)
        if cusum_newly_triggered:
            self.detected = self.cusum_terminated = True
            reward -= self.lmbda
        if hard_abort:
            self.detected = True  # hard abort is also an observable detection event
            reward -= self.cfg.abort_penalty / self.cfg.damage_scale
            self.hard_abort = self.done = True
            self.termination_reason = "hard_abort"
        elif self.round >= self.cfg.max_rounds:
            self.done = True
            self.termination_reason = "horizon_reached"

        feat = self._make_feature_vec(qber_hat, traffic_obs, cusum_stat)
        self.frame_buf.append(feat)
        info = dict(round=self.round, regime=int(regime), value_class=int(value_class),
            true_qber=true_qber, qber_hat=qber_hat, pooled_qber=pooled_qber,
            f_actual=f_actual, secure_bits_generated=secure_bits_generated,
            demand_bits=demand_bits, exposure_weighted_consumed=exposure_weighted,
            deficit=deficit, shadow_deficit=shadow_deficit, availability_damage=damage,
            cusum_stat=cusum_stat, hard_abort=hard_abort, detected=self.detected,
            cusum_detected=self.cusum_terminated,
            cusum_newly_triggered=cusum_newly_triggered,
            termination_reason=self.termination_reason,
            budget_remaining=self.budget_remaining, pool_level=self.ledger.pool_level(),
            pool_overflow=self.ledger.total_overflow,
            metadata_overhead=metadata_overhead,
            metadata_throughput_loss=metadata_throughput_loss,
            metadata_latency=metadata_latency,
            traffic_rate_obs=traffic_obs["traffic_rate_obs"],
            burstiness_obs=traffic_obs["burstiness_obs"],
            size_obs=traffic_obs["size_obs"])
        self._log.append(info)
        if not self.done:
            self.hidden.step()
        return self._stacked_obs(), reward, self.done, info


In [ ]:
# %% Cell 9 — Baseline policies
class RandomEvePolicy:
    """Spends budget with no regard for state; picks a uniformly random action
    each round (budget-blind, information-blind)."""
    def __init__(self, n_actions, rng):
        self.n_actions = n_actions
        self.rng = rng

    def act(self, obs, raw_feat_dict=None):
        return self.rng.integers(0, self.n_actions)

class GreedyTrafficEvePolicy:
    """
    Reactive rule combining BOTH signals, as originally specified in the
    baseline design table (QBER-aware + traffic-aware, no RL/planning):
      1. Safety throttle: if the current observed QBER is already
         uncomfortably close to the abort threshold, back off completely,
         regardless of traffic level (basic self-preservation).
      2. Otherwise, attack harder when current observed traffic rate is high
         (quantile-thresholded against a calibration sample).
    This is NOT anticipatory (no memory/planning) — it only reacts to the
    current round's observation, which is exactly the property we want to
    contrast against the RL agents later.
    """
    def __init__(self, intercept_levels, qber_abort, caution_frac=0.7):
        self.intercept_levels = intercept_levels
        self.thresholds = None
        self.qber_caution_level = 0.025  # reachable under the revised action grid

    def calibrate(self, traffic_rate_samples):
        qs = np.linspace(0, 1, len(self.intercept_levels) + 1)[1:-1]
        self.thresholds = np.quantile(traffic_rate_samples, qs)

    def act(self, obs, raw_feat_dict):
        assert self.thresholds is not None, "call calibrate() first"
        qber_hat = raw_feat_dict["qber_hat"]
        if qber_hat >= self.qber_caution_level:
            return 0  # back off entirely regardless of traffic
        rate = raw_feat_dict["traffic_rate_obs"]
        action_idx = int(np.searchsorted(self.thresholds, rate))
        return action_idx


class BudgetMatchedPolicy:
    """Constant-rate baseline using the same total interception budget."""
    def __init__(self, levels, total_budget, horizon, qubits_per_round):
        self.levels = np.asarray(levels, dtype=float)
        self.target = min(total_budget / max(horizon * qubits_per_round, 1), self.levels[-1])
    def act(self, obs, raw_feat_dict=None):
        return int(np.argmin(np.abs(self.levels - self.target)))


class DetectorAwarePolicy:
    uses_raw_features = True
    """Throttles to the largest action predicted to remain below the calibrated CUSUM limit."""
    def __init__(self, levels, q0, slack, threshold, margin=0.25):
        self.levels, self.q0, self.slack = tuple(levels), q0, slack
        self.threshold, self.margin = threshold, margin
    def act(self, obs, raw_feat_dict):
        stat = raw_feat_dict.get("cusum_stat", 0.0)
        safe = [i for i, f in enumerate(self.levels)
                if stat + max(0.0, f * QBER_SLOPE - self.slack) < self.threshold * (1-self.margin)]
        return max(safe) if safe else 0


class TimeOnlyPolicy:
    uses_raw_features = True
    """Clock-only periodic schedule, with no traffic input."""
    def __init__(self, max_rounds, attack_index):
        self.max_rounds, self.attack_index = max_rounds, attack_index
    def act(self, obs, raw_feat_dict=None):
        round_index = int(raw_feat_dict.get("round", 0)) if raw_feat_dict else 0
        return self.attack_index if round_index % 10 == 0 else 0


class StateOraclePolicy:
    """State-informed reference; optionally follows the state/budget QMDP plan."""
    def __init__(self, highest_action, q_values=None, budget_step=None):
        self.highest_action = int(highest_action)
        self.q_values = q_values
        self.budget_step = budget_step
    def bind(self, env): self.env = env
    def act(self, obs, raw_feat_dict=None):
        if self.q_values is not None:
            t = min(self.env.round, self.env.cfg.max_rounds - 1)
            remaining_rounds = (self.env.budget_remaining /
                                self.env.cfg.n_qubits_per_round)
            step = self.budget_step or min(f for f in self.env.cfg.intercept_levels if f > 0)
            b = int(np.clip(np.floor(remaining_rounds / step + 1e-9),
                            0, self.q_values.shape[2] - 1))
            state_i = JOINT_STATES.index((self.env.hidden.regime, self.env.hidden.value))
            return int(np.argmax(self.q_values[t, state_i, b, :]))
        # Transparent heuristic used for the delay-50 preflight: attack every
        # HIGH-value round, independent of whether the regime is ACTIVE or CALM.
        return self.highest_action if self.env.hidden.value == ValueClass.HIGH else 0


class MetadataHMMPolicy:
    """Six-state forward filter using metadata emissions fitted from the simulator."""
    uses_raw_features = True
    def __init__(self, cfg, emission_mean, emission_std, threshold=0.25):
        self.cfg, self.mean, self.std, self.threshold = cfg, emission_mean, emission_std, threshold
        self.belief = STATIONARY_JOINT.copy()
        self.high_action = len(cfg.intercept_levels)-1
    def reset(self): self.belief = STATIONARY_JOINT.copy()
    def _likelihood(self, feature):
        values = np.asarray([feature.get(k, 0.0) for k in
            ("traffic_rate_obs", "burstiness_obs", "size_obs")], dtype=float)
        per_class = np.exp(-0.5*np.sum(((values-self.mean)/self.std)**2 +
            np.log(2*np.pi*self.std**2), axis=1))
        marginal = float(STATIONARY_VALUE @ per_class)
        return np.asarray([self.cfg.rho*per_class[int(v)] + (1-self.cfg.rho)*marginal
                           for _,v in JOINT_STATES])
    def act(self, obs, raw_feat_dict):
        if raw_feat_dict.get("round", 0) > 0:
            likelihood = self._likelihood(raw_feat_dict)  # metadata from the previous state
            posterior = self.belief * likelihood
            z = posterior.sum()
            posterior = posterior/z if z > 0 else self.belief
            self.belief = posterior @ JOINT_TRANS  # predict the current hidden state
        p_high = sum(self.belief[i] for i,(_,v) in enumerate(JOINT_STATES) if v == ValueClass.HIGH)
        if raw_feat_dict.get("budget_frac", 0.0) <= 0 or raw_feat_dict.get("cusum_stat",0.0) >= self.cfg.cusum_threshold*.75:
            return 0
        return self.high_action if p_high >= self.threshold else 0


def fit_metadata_emissions(cfg, samples_per_class=3000, seed=412):
    """Fit diagonal Gaussian emissions from independent synthetic traffic traces."""
    means, stds = [], []
    for value in ValueClass:
        traffic = TrafficGenerator(np.random.default_rng(seed+int(value)),
            window=cfg.window, metadata_noise_std=cfg.metadata_noise_std)
        traffic.reset(); rows=[]
        for t in range(samples_per_class + 10):
            _, feat = traffic.step(value)
            if t >= 10:
                rows.append([feat["traffic_rate_obs"], feat["burstiness_obs"], feat["size_obs"]])
        arr=np.asarray(rows, dtype=float)
        means.append(arr.mean(axis=0)); stds.append(np.maximum(arr.std(axis=0), 1e-3))
    return np.asarray(means), np.asarray(stds)


def metadata_predictive_logloss(cfg, emission_mean, emission_std, episodes=200, seed0=20000):
    """Compare state prediction log-loss with metadata filtering and the stationary null."""
    meta_losses, null_losses = [], []
    for ep in range(episodes):
        env=EveEnv(cfg); env.reset(seed0+ep)
        posterior=STATIONARY_JOINT.copy()
        for _ in range(cfg.max_rounds):
            _,_,done,info=env.step(0)
            prior=posterior @ JOINT_TRANS
            state_idx=JOINT_STATES.index((Regime(info["regime"]), ValueClass(info["value_class"])))
            null_losses.append(-np.log(max(STATIONARY_JOINT[state_idx],1e-12)))
            meta_losses.append(-np.log(max(prior[state_idx],1e-12)))
            obs={k:info[k] for k in ("traffic_rate_obs","burstiness_obs","size_obs")}
            likelihood=MetadataHMMPolicy(cfg,emission_mean,emission_std)._likelihood(obs)
            post=prior*likelihood
            posterior=post/post.sum() if post.sum()>0 else prior
            if done: break
    return dict(metadata_logloss=float(np.mean(meta_losses)),
                null_logloss=float(np.mean(null_losses)),
                delta_logloss=float(np.mean(null_losses)-np.mean(meta_losses)))


class ApproximateQMDPPolicy:
    """QMDP over hidden demand state and quantized attack budget.

    It uses expected one-round incremental unmet demand and empirical detection
    costs. Pool carry-over and detector state are omitted, so this is a reduced
    planning reference, not an exact solution to the full simulator.
    """
    uses_raw_features = True
    def __init__(self, cfg, emission_mean, emission_std, q_values, budget_step):
        self.cfg, self.q_values, self.budget_step = cfg, q_values, budget_step
        self.filter = MetadataHMMPolicy(cfg, emission_mean, emission_std, threshold=0.0)
    def act(self, obs, raw_feat_dict):
        self.filter.act(obs, raw_feat_dict)
        t = min(int(raw_feat_dict.get("round", 0)), self.cfg.max_rounds-1)
        remaining_rounds = raw_feat_dict.get("budget_frac", 1.0) * (
            self.cfg.total_budget_qubits / self.cfg.n_qubits_per_round)
        b = int(np.clip(np.floor(remaining_rounds / self.budget_step + 1e-9),
            0, self.q_values.shape[2]-1))
        scores = self.filter.belief @ self.q_values[t, :, b, :]
        return int(np.argmax(scores))


def build_qmdp_values(cfg, emission_risk_curve, risk_penalty=1.0, budget_step=None, gamma=0.999):
    """Vectorized finite-horizon QMDP backup over hidden state and attack budget.

    Budget units match the smallest nonzero action by default. This avoids
    charging every 0.005 action as a full 0.02 bucket, while flooring the
    observed remaining budget in ApproximateQMDPPolicy prevents overspending.
    """
    n_actions = len(cfg.intercept_levels)
    n_states = len(JOINT_STATES)
    levels = np.asarray(cfg.intercept_levels, dtype=float)
    positive_actions = levels[levels > 0]
    if budget_step is None:
        budget_step = float(positive_actions.min()) if len(positive_actions) else 0.01
    if budget_step <= 0:
        raise ValueError("budget_step must be positive")

    budget_total = cfg.total_budget_qubits / cfg.n_qubits_per_round
    n_budget = int(np.floor(budget_total / budget_step + 1e-9))
    budget_indices = np.arange(n_budget + 1)
    action_costs = np.ceil(levels / budget_step - 1e-12).astype(int)
    values = np.zeros((cfg.max_rounds + 1, n_states, n_budget + 1), dtype=np.float32)
    q_values = np.full((cfg.max_rounds, n_states, n_budget + 1, n_actions),
                       -1e9, dtype=np.float32)

    expected_sifted = cfg.n_qubits_per_round * 0.5
    usable = max(0.0, expected_sifted - cfg.sample_size_qber)
    supply0 = usable * secure_key_rate(cfg.qber_baseline)
    risk_f = np.asarray(emission_risk_curve[0], dtype=float)
    risk_p = np.asarray(emission_risk_curve[1], dtype=float)
    p_null = float(np.interp(0.0, risk_f, risk_p))

    demands = np.asarray([CLASS_PARAMS[value_class]["rate"] * cfg.key_bits_per_message
                          for _, value_class in JOINT_STATES], dtype=float)
    base_short = np.maximum(0.0, demands - supply0)
    action_supply = np.asarray([
        usable * secure_key_rate(min(1.0, cfg.qber_baseline + f * QBER_SLOPE))
        for f in levels
    ])
    damage = np.maximum(0.0, demands[:, None] - action_supply[None, :]) - base_short[:, None]
    detection_penalty = np.maximum(0.0, np.interp(levels, risk_f, risk_p) - p_null)
    immediate = damage / cfg.damage_scale - risk_penalty * detection_penalty[None, :]

    for t in range(cfg.max_rounds - 1, -1, -1):
        current_values = np.full((n_states, n_budget + 1), -1e9, dtype=np.float32)
        for a, cost in enumerate(action_costs):
            remaining_idx = np.maximum(0, budget_indices - cost)
            continuation = JOINT_TRANS @ values[t + 1][:, remaining_idx]
            scores = immediate[:, a, None] + gamma * continuation
            scores[:, budget_indices < cost] = -1e9
            q_values[t, :, :, a] = scores
            current_values = np.maximum(current_values, scores)
        values[t] = current_values
    return q_values


In [ ]:
# %% Cell 10 — Evaluation harness
def run_episode(cfg: EnvConfig, policy, seed, is_greedy=False):
    env = EveEnv(cfg)
    obs = env.reset(seed=seed)
    if hasattr(policy, "bind"):
        policy.bind(env)
    total_reward = 0.0
    done = False
    while not done:
        if is_greedy or getattr(policy, "uses_raw_features", False):
            last_feat = env.frame_buf[-1]
            raw_feat_dict = dict(zip(env.feature_names, last_feat))
            raw_feat_dict["cusum_stat"] = env.detector.s
            raw_feat_dict["round"] = env.round
            raw_feat_dict["budget_frac"] = env.budget_remaining / max(cfg.total_budget_qubits, 1e-12)
            action = policy.act(obs, raw_feat_dict)
        else:
            action = policy.act(obs)
        obs, reward, done, info = env.step(action)
        total_reward += reward

    log = pd.DataFrame(env._log)
    metrics = dict(
        total_damage_reward=total_reward,
        total_unweighted_damage_reward=total_reward,
        total_qubits_intercepted=cfg.total_budget_qubits - env.budget_remaining,
        budget_used_frac=(cfg.total_budget_qubits - env.budget_remaining) / cfg.total_budget_qubits,
        mean_true_qber=log["true_qber"].mean(),
        max_true_qber=log["true_qber"].max(),
        hard_abort=bool(env.hard_abort),
        cusum_terminated=bool(env.cusum_terminated),
        detected=bool(log["detected"].any()),
        termination_reason=env.termination_reason,
        cusum_max=log["cusum_stat"].max(),
        cusum_detected=bool(log["cusum_detected"].any()),
        detection_round=(float(log.loc[log["detected"], "round"].min())
                         if log["detected"].any() else np.nan),
        total_secure_bits_generated=log["secure_bits_generated"].sum(),
        total_exhaustion_deficit=log["deficit"].sum(),
        total_availability_damage=log["availability_damage"].sum(),
        mean_pool_overflow=log["pool_overflow"].mean(),
        metadata_overhead=float(log["metadata_overhead"].mean()),
        metadata_throughput_loss=float(log["metadata_throughput_loss"].mean()),
        metadata_latency=float(log["metadata_latency"].mean()),
        episode_length=len(log),
    )
    return metrics, log


def run_batch(cfg: EnvConfig, policy_factory, n_episodes=30, seed0=0, is_greedy=False):
    rows = []
    for i in range(n_episodes):
        policy = policy_factory(seed0 + i)
        metrics, _ = run_episode(cfg, policy, seed0 + i, is_greedy=is_greedy)
        rows.append(metrics)
    return pd.DataFrame(rows)




In [ ]:
# %% Cell 11 (REPLACED) — Sanity checks
cfg = EnvConfig(seed=0)

# Explicit calibration with diagnostics (see note in calibrate_cusum_threshold docstring)
cfg.cusum_threshold, cusum_diag = calibrate_cusum_threshold(
    cfg.qber_baseline, cfg.sample_size_qber, cfg.max_rounds, cfg.cusum_slack,
    n_calib_episodes=20000, seed=999, return_diagnostics=True)
print(f"[sanity] calibrated CUSUM threshold: {cfg.cusum_threshold:.5f}")
print(f"[sanity] null-process max-stat diagnostics: {cusum_diag}")
print("[expect] frac_exactly_zero should be well below 1.0 (e.g. <0.7), and "
      "p50/p75/p90 should be spread out rather than all clustered near 0 -- "
      "otherwise the threshold is still degenerate.")

# NEW: explicit supply/demand balance check before running any episodes
supply0 = baseline_secure_bits_per_round(cfg)
print(f"\n[calibration check] baseline secure bits/round (f=0): {supply0:.1f}")
for vc in ValueClass:
    demand = CLASS_PARAMS[vc]["rate"] * cfg.key_bits_per_message
    print(f"  {vc.name:8s} demand/round: {demand:7.1f}  "
          f"net (supply-demand): {supply0-demand:+7.1f}")
mean_demand = sum(STATIONARY_VALUE[int(vc)] * CLASS_PARAMS[vc]["rate"]
                   * cfg.key_bits_per_message for vc in ValueClass)
print(f"[stationary load] E[demand]/E[no-attack supply]={mean_demand/supply0:.3f}; "
      f"requested load={cfg.load:.3f}; sweep load explicitly in the operating-regime study.")

# Metadata must show no predictive gain at rho=0 before expensive PPO experiments.
emission_mean, emission_std = fit_metadata_emissions(cfg)
rho_audit = {}
for rho in (0.0, 1.0):
    cfg_rho = EnvConfig(load=cfg.load, rho=rho)
    rho_audit[rho] = metadata_predictive_logloss(cfg_rho, emission_mean, emission_std,
        episodes=200, seed0=20000)
print("\n=== Metadata predictive value audit (positive delta_LL means better prediction) ===")
print(pd.DataFrame(rho_audit).T.rename_axis("rho").to_string())
if abs(rho_audit[0.0]["delta_logloss"]) > 0.05:
    print("[audit warning] rho=0 still shows predictive gain; inspect the emission model before PPO.")

# --- Random Eve ---
def make_random_policy(seed):
    return RandomEvePolicy(len(cfg.intercept_levels), np.random.default_rng(seed))

df_random = run_batch(cfg, make_random_policy, n_episodes=30, seed0=100)
print("\n=== Random Eve summary (30 episodes) ===")
print(df_random.describe().T[["mean", "std", "min", "max"]])
print(f"[explicit] Random Eve: hard_abort rate = {df_random['hard_abort'].mean():.3f}, "
      f"cusum_detected rate = {df_random['cusum_detected'].mean():.3f}")

# --- Calibrate & run Greedy Eve ---
calib_env = EveEnv(cfg)
calib_env.reset(seed=777)
rate_samples = []
for _ in range(cfg.max_rounds):
    _, _, done, info = calib_env.step(0)
    last_feat = calib_env.frame_buf[-1]
    rate_samples.append(dict(zip(calib_env.feature_names, last_feat))["traffic_rate_obs"])
    if done:
        break

greedy_policy_template = GreedyTrafficEvePolicy(cfg.intercept_levels, cfg.qber_abort)
greedy_policy_template.calibrate(np.array(rate_samples))
print(f"\n[sanity] greedy thresholds on traffic_rate_obs: {greedy_policy_template.thresholds}")

def make_greedy_policy(seed):
    return greedy_policy_template

df_greedy = run_batch(cfg, make_greedy_policy, n_episodes=30, seed0=100, is_greedy=True)
print("\n=== Greedy Traffic Eve summary (30 episodes) ===")
print(df_greedy.describe().T[["mean", "std", "min", "max"]])
print(f"[explicit] Greedy Eve: hard_abort rate = {df_greedy['hard_abort'].mean():.3f}, "
      f"cusum_detected rate = {df_greedy['cusum_detected'].mean():.3f}")

# --- Theoretical cross-checks ---
mean_f_random_unconstrained = np.mean(cfg.intercept_levels)
print(f"\n[check] mean intercept level if budget unconstrained: {mean_f_random_unconstrained:.3f}")
print(f"[check] implied instantaneous QBER at that f: "
      f"{cfg.qber_baseline + mean_f_random_unconstrained*QBER_SLOPE:.4f} (abort at {cfg.qber_abort})")
print("[check] -> episodes should now run MUCH longer (tens to ~300 rounds), "
      "with hard_abort becoming rare-ish rather than near-certain within 2-3 rounds.")

# --- Single-episode diagnostic plot (Random Eve) ---
policy = make_random_policy(seed=42)
_, log = run_episode(cfg, policy, seed=42)

fig, axes = plt.subplots(5, 1, figsize=(10, 12), sharex=True)
axes[0].plot(log["round"], log["true_qber"], label="true QBER")
axes[0].plot(log["round"], log["qber_hat"], alpha=0.4, label="sampled QBER (per-round)")
axes[0].plot(log["round"], log["pooled_qber"], label="pooled QBER (abort decision)", lw=2)
axes[0].axhline(cfg.qber_abort, color="red", ls="--", label="abort threshold")
axes[0].legend(fontsize=8); axes[0].set_ylabel("QBER")

regime_colors = log["regime"].map({0: "white", 1: "lightblue"})
for i in range(len(log) - 1):
    axes[1].axvspan(log["round"].iloc[i], log["round"].iloc[i+1], color=regime_colors.iloc[i], alpha=0.5)
axes[1].plot(log["round"], log["value_class"], drawstyle="steps-post")
axes[1].set_ylabel("Value class\n[shading=ACTIVE]")

axes[2].plot(log["round"], log["pool_level"])
axes[2].set_ylabel("Key pool (bits)")

axes[3].plot(log["round"], log["deficit"])
axes[3].set_ylabel("Exhaustion\ndeficit (bits)")

axes[4].plot(log["round"], log["budget_remaining"])
axes[4].set_ylabel("Budget left\n(qubits)")
axes[4].set_xlabel("round")

plt.tight_layout()
plt.savefig("sanity_random_eve_trace_v2.png", dpi=120)
plt.show()
print("\nSaved: sanity_random_eve_trace_v2.png")


In [ ]:
# %% Cell 11b (REPLACED) — Constant-action sweep: empirically validate the CUSUM breakeven point
# Empirical detector characterization only: noisy finite-sample paths do not have a sharp theoretical f*.
print(f"Empirical null calibration target: {cfg.target_false_alarm:.1%} false alarms over {cfg.max_rounds} rounds")

class ConstantActionPolicy:
    """Diagnostic-only: always plays the same fixed action index every round."""
    def __init__(self, action_idx):
        self.action_idx = action_idx

    def act(self, obs):
        return self.action_idx

rows = []
for action_idx, f_level in enumerate(cfg.intercept_levels):
    def make_policy(seed, action_idx=action_idx):
        return ConstantActionPolicy(action_idx)
    df = run_batch(cfg, make_policy, n_episodes=30, seed0=200)
    rows.append(dict(
        action_idx=action_idx, f_level=f_level,
        mean_episode_length=df["episode_length"].mean(),
        frac_horizon_reached=(df["termination_reason"] == "horizon_reached").mean(),
        frac_cusum_terminated=df["detected"].mean(),
        frac_hard_abort=(df["termination_reason"] == "hard_abort").mean(),
        mean_budget_used_frac=df["budget_used_frac"].mean(),
        mean_reward=df["total_damage_reward"].mean(),
    ))

breakeven_table = pd.DataFrame(rows)
print("\n=== Constant-action empirical detection validation (30 episodes each) ===")
print(breakeven_table.to_string(index=False))
print(f"\n[check] f=0 should have a CUSUM false-alarm rate near {cfg.target_false_alarm:.1%}; finite-sample results vary.")
print("Use the measured detection rates and episode lengths above; this empirical sweep assumes no theoretical breakeven f*.")


In [ ]:
# %% Cell 11c — NEW: empirical detection probability vs sustained attack
# Characterizes CUSUM detection probability and detection timing as a function of sustained
# interception fraction f, independent of the discrete action grid. This is
# the "detector operating characteristic curve" -- validates the sharp
# knife-edge transition at f* and quantifies exactly how fast ARL collapses
# once f exceeds breakeven. Also useful as a standalone diagnostic
# figure/table for the eventual writeup.
import copy
from scipy.stats import beta as beta_dist

f_sweep = np.concatenate([
    np.linspace(0.0, 0.04, 9),
])
f_sweep = np.unique(np.round(f_sweep, 4))

n_episodes_arl = 2000
arl_rows = []
for f in f_sweep:
    cfg_f = copy.deepcopy(cfg)
    cfg_f.intercept_levels = (f,)   # single always-on action, bypasses discrete grid entirely

    def make_const_policy(seed, f=f):
        return ConstantActionPolicy(0)

    df_f = run_batch(cfg_f, make_const_policy, n_episodes=n_episodes_arl, seed0=300)
    arl_rows.append(dict(
        f=f,
        mean_episode_length=df_f["episode_length"].mean(),
        detections=int(df_f["cusum_detected"].sum()),
        mean_detection_round=float(df_f["detection_round"].mean()),
        frac_cusum_terminated=float(df_f["cusum_detected"].mean()),
        frac_horizon_reached=(df_f["termination_reason"] == "horizon_reached").mean(),
        frac_hard_abort=(df_f["termination_reason"] == "hard_abort").mean(),
    ))

for row in arl_rows:
    k, n = row["detections"], n_episodes_arl
    row["detection_ci_lo"] = 0.0 if k == 0 else float(beta_dist.ppf(0.025, k, n-k+1))
    row["detection_ci_hi"] = 1.0 if k == n else float(beta_dist.ppf(0.975, k+1, n-k))
arl_df = pd.DataFrame(arl_rows)
print("=== Empirical CUSUM detection rates with 95% Clopper-Pearson intervals ===")
print(arl_df.to_string(index=False))

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(arl_df["f"], arl_df["mean_episode_length"], marker="o")
axes[0].set_ylabel("Mean episode length\n(rounds to termination)")

axes[1].errorbar(arl_df["f"], arl_df["frac_cusum_terminated"],
    yerr=[arl_df["frac_cusum_terminated"]-arl_df["detection_ci_lo"],
          arl_df["detection_ci_hi"]-arl_df["frac_cusum_terminated"]],
    marker="o", color="darkorange", capsize=3)
axes[1].set_ylabel("Fraction detected\nby CUSUM")
axes[1].set_xlabel("sustained interception fraction f")

plt.tight_layout()
plt.savefig("cusum_detection_operating_characteristic.png", dpi=120)
plt.show()
print("\nSaved: cusum_detection_operating_characteristic.png")
print("Detection rates and intervals are empirical; do not interpret them as a drift boundary.")


In [ ]:
# %% Cell 11d — NEW: expected-value sanity check across constant strategies
# Confirms rescaled penalties allow a non-trivial policy to plausibly beat
# "always play safe" (action 0), i.e. that the reward landscape isn't
# degenerate before we invest in RL training.
rows = []
for action_idx, f_level in enumerate(cfg.intercept_levels):
    def make_policy(seed, action_idx=action_idx):
        return ConstantActionPolicy(action_idx)
    df = run_batch(cfg, make_policy, n_episodes=60, seed0=400)
    rows.append(dict(
        f_level=f_level,
        mean_reward=df["total_damage_reward"].mean(),
        std_reward=df["total_damage_reward"].std(),
        mean_episode_length=df["episode_length"].mean(),
        frac_survived_full_horizon=(df["termination_reason"] == "horizon_reached").mean(),
    ))

sanity_df = pd.DataFrame(rows)
print("=== Expected-value sanity check across constant strategies (60 episodes each) ===")
print(sanity_df.to_string(index=False))
print("\n[expect] at least one f>0 row should now have mean_reward competitive with "
      "or exceeding f=0's mean_reward -- confirming attacking isn't strictly "
      "dominated by caution under the rescaled penalty.")

# Delay-50 preflight: do not train the full grid if policies still have no measurable damage signal.
preflight_cfg = EnvConfig(obs_mode="full", delay_rounds=50, warm_start_rounds=5.0)
preflight_oracle = run_batch(preflight_cfg,
    lambda seed: StateOraclePolicy(len(preflight_cfg.intercept_levels)-1),
    n_episodes=30, seed0=55000)
preflight_constant_rows = []
for action_idx, f_level in enumerate(preflight_cfg.intercept_levels):
    batch = run_batch(preflight_cfg, lambda seed, a=action_idx: ConstantActionPolicy(a),
                      n_episodes=30, seed0=55000)
    preflight_constant_rows.append((f_level, float(batch["total_availability_damage"].mean()),
                                    float(batch["total_damage_reward"].mean())))
best_preflight = max(preflight_constant_rows, key=lambda row: row[2])
oracle_preflight_damage = float(preflight_oracle["total_availability_damage"].mean())
print("\n=== Delay-50 signal preflight (30 paired episodes) ===")
print(f"StateOracle mean incremental unmet demand: {oracle_preflight_damage:.3f} bits")
print(f"Best constant f={best_preflight[0]:g}: mean incremental unmet demand={best_preflight[1]:.3f} bits")
if oracle_preflight_damage <= 0 and best_preflight[1] <= 0:
    print("[STOP] No delay-50 attack signal under this configuration; adjust load/capacity/warm start before the RL grid.")
else:
    print("[PASS] At least one informed/constant policy causes measurable incremental damage at delay=50.")


In [ ]:
# %% Cell 12 — Gymnasium wrapper around EveEnv
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.callbacks import BaseCallback

# Evaluation seeds used by calibration and later experiment cells. Define these
# before the training/evaluation calls so sequential notebook execution can resolve them.
VALIDATION_SEED0 = 5000
TEST_SEED0 = 100000


class EveGymEnv(gym.Env):
    """
    Thin gymnasium.Env adapter around the validated EveEnv core. All physics/
    detection/hidden-state logic stays in EveEnv untouched; this wrapper only
    translates the calling convention and handles the terminated-vs-truncated
    distinction, which matters for correct value bootstrapping in PPO:

      - terminated=True : hard_abort (physical protocol abort)
      - truncated=True  : horizon_reached (an externally imposed session
        length limit -- more reward would have been possible with a longer
        horizon, so PPO should bootstrap the value function here rather than
        treating it as a true terminal state)
    """
    metadata = {"render_modes": []}

    def __init__(self, cfg: EnvConfig):
        super().__init__()
        self.core = EveEnv(cfg)
        self.action_space = spaces.Discrete(self.core.n_actions)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.core.obs_dim,), dtype=np.float32
        )

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)   # satisfies gymnasium's base-class seeding contract (sets self._np_random)
        obs = self.core.reset(seed=seed)  # actual simulator RNG/hidden-state reseeding happens here, unchanged
        return obs, {}

    def set_lambda(self, value):
        self.core.lmbda = float(value)

    def step(self, action):
        obs, reward, done, info = self.core.step(action)
        terminated = bool(done and info["termination_reason"] == "hard_abort")
        truncated = bool(done and info["termination_reason"] == "horizon_reached")
        return obs, float(reward), terminated, truncated, info


class LagrangeCallback(BaseCallback):
    """Dual update for the episode-level detection constraint."""
    def __init__(self, delta=0.05, lr=0.5, n_episodes=64, initial_lambda=1.0):
        super().__init__()
        self.delta, self.lr, self.n_episodes = float(delta), float(lr), int(n_episodes)
        self.lmbda = float(initial_lambda)
        self.detection_buffer, self.trajectory = [], []

    def _on_step(self):
        for info, done in zip(self.locals["infos"], self.locals["dones"]):
            if done:
                self.detection_buffer.append(float(info.get("detected", False)))
        if len(self.detection_buffer) >= self.n_episodes:
            rate = float(np.mean(self.detection_buffer))
            self.lmbda = max(0.0, self.lmbda + self.lr * (rate - self.delta))
            self.training_env.env_method("set_lambda", self.lmbda)
            self.logger.record("lagrange/lambda", self.lmbda)
            self.logger.record("lagrange/detection_rate", rate)
            self.trajectory.append(dict(episodes=len(self.trajectory)*self.n_episodes+self.n_episodes,
                                        detection_rate=rate, lmbda=self.lmbda))
            self.detection_buffer.clear()
        return True



class RecurrentPolicyAdapter:
    def __init__(self, model, vec_env):
        self.model, self.vec_env = model, vec_env
        self.state, self.episode_start = None, True
    def act(self, obs):
        obs_in = self.vec_env.normalize_obs(np.asarray(obs,dtype=np.float32).reshape(1,-1))
        action, self.state = self.model.predict(obs_in, state=self.state,
            episode_start=np.asarray([self.episode_start]), deterministic=True)
        self.episode_start = False
        return int(np.asarray(action).reshape(-1)[0])


def train_recurrent_ppo(cfg, total_timesteps=400_000, n_envs=8,
                        validation_episodes=200, test_episodes=2000, seed=1):
    """Train/evaluate RecurrentPPO with disjoint validation and test episodes."""
    from sb3_contrib import RecurrentPPO
    set_random_seed(seed)
    vec_env = make_training_vec_env_v2(cfg.obs_mode, seed, cfg.delay_rounds,
        n_envs=n_envs, detection_limit=cfg.detection_limit)
    model = RecurrentPPO("MlpLstmPolicy", vec_env, learning_rate=3e-4,
        n_steps=512, batch_size=256, n_epochs=10, gamma=0.999,
        gae_lambda=0.95, ent_coef=0.01, verbose=0, seed=seed, device=PPO_DEVICE)
    callback = LagrangeCallback(delta=cfg.detection_limit)
    model.learn(total_timesteps=total_timesteps, callback=callback)
    factory = lambda _seed: RecurrentPolicyAdapter(model, vec_env)
    validation = run_batch(cfg, factory, validation_episodes, seed0=VALIDATION_SEED0)
    test = run_batch(cfg, factory, test_episodes, seed0=TEST_SEED0)
    summary = dict(validation_damage=validation.total_damage_reward.mean(),
        validation_detection=validation.detected.mean(),
        test_damage=test.total_damage_reward.mean(), test_detection=test.detected.mean(),
        feasible=bool(test.detected.mean() <= cfg.detection_limit),
        lagrange_lambda=callback.lmbda, lagrange_trajectory=callback.trajectory)
    return model, vec_env, summary, test


In [ ]:
# %% Cell 13 — Environment API sanity check
from gymnasium.utils.env_checker import check_env

cfg_check = EnvConfig(seed=0, obs_mode="full")
gym_env_check = EveGymEnv(cfg_check)
check_env(gym_env_check, warn=True, skip_render_check=True)
print("[sanity] gymnasium check_env passed with no exceptions.")



In [ ]:
# %% Cell 14 — Vec-env construction and PPO training helper
def make_training_vec_env(obs_mode: str, base_seed: int, delay_rounds: int = 0,
                           normalize_obs=True, normalize_reward=True):
    """
    Single-process DummyVecEnv (fine here -- this is a pure-Python analytic
    simulator, not a physics-heavy sim) wrapped in VecNormalize to handle the
    very different feature scales in the 'full' observation (qber_hat ~0.02-
    0.1, size_obs ~100-900, budget_frac 0-1, etc.) and the wide reward range
    we measured (std up to ~168 at f=0.02).

    IMPORTANT: SB3's auto-reset calls env.reset() with no seed at episode
    boundaries. Per EveEnv.reset()'s semantics, seed=None means the RNG and
    hidden Markov process are NOT reseeded between episodes -- they continue
    as one long stochastic stream. This is intentional: it gives the agent
    broad exposure to many different hidden-state trajectories over a long
    training run rather than replaying a small fixed set of seeded episodes.
    Evaluation, by contrast, always uses explicit fixed seeds (see
    evaluate_rl_policy below) so comparisons against the baselines remain
    apples-to-apples.
    """
    def _make():
        cfg = EnvConfig(seed=base_seed, obs_mode=obs_mode, delay_rounds=delay_rounds)
        env = EveGymEnv(cfg)
        env = Monitor(env)
        return env

    vec_env = DummyVecEnv([_make])
    vec_env = VecNormalize(vec_env, norm_obs=normalize_obs, norm_reward=normalize_reward,
                            clip_obs=10.0, clip_reward=20.0, gamma=0.995)
    return vec_env


def train_ppo_agent(obs_mode: str, total_timesteps: int, seed: int,
                     delay_rounds: int = 0, save_path: str = None, verbose=1):
    set_random_seed(seed)
    vec_env = make_training_vec_env(obs_mode, base_seed=seed, delay_rounds=delay_rounds)

    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.995,        # slightly above SB3 default 0.99 -- episodes run up to 300 rounds
        gae_lambda=0.95,
        ent_coef=0.01,      # small entropy bonus: encourage exploration given delayed/sparse detection risk
        verbose=verbose,
        seed=seed,
        policy_kwargs=dict(net_arch=[64, 64]),
        device=PPO_DEVICE,
    )
    model.learn(total_timesteps=total_timesteps)

    if save_path is not None:
        model.save(save_path)
        vec_env.save(save_path + "_vecnormalize.pkl")

    return model, vec_env


In [ ]:
# %% Cell 15 — SMOKE TEST: short training runs
if SESSION_STAGE == 'all':
    # NOT final results -- purely confirming the plumbing works end-to-end and
    # some learning signal is present. Real experiments need far more timesteps
    # (see Phase 2 continuation).
    SMOKE_TEST_TIMESTEPS = 20_000

    print("=== Smoke test: QBER-only agent ===")
    model_qber_smoke, vecenv_qber_smoke = train_ppo_agent(
        obs_mode="qber_budget", total_timesteps=SMOKE_TEST_TIMESTEPS, seed=1,
        delay_rounds=0, save_path=None, verbose=1
    )

    print("\n=== Smoke test: Messaging-aware agent ===")
    model_full_smoke, vecenv_full_smoke = train_ppo_agent(
        obs_mode="full", total_timesteps=SMOKE_TEST_TIMESTEPS, seed=1,
        delay_rounds=0, save_path=None, verbose=1
    )



In [ ]:
# %% Cell 16 — RL policy adapter + evaluation via the EXISTING run_batch harness
class RLPolicyAdapter:
    """
    Wraps a trained SB3 model (+ its frozen VecNormalize observation stats)
    so it plugs directly into run_episode/run_batch alongside Random/Greedy/
    ConstantAction policies -- guaranteeing IDENTICAL evaluation methodology
    (same seeded episodes, same metrics) across every agent type.
    """
    def __init__(self, model, vec_normalize: VecNormalize = None, deterministic=True):
        self.model = model
        self.vec_normalize = vec_normalize
        self.deterministic = deterministic

    def act(self, obs):
        obs_in = obs.reshape(1, -1).astype(np.float32)
        if self.vec_normalize is not None:
            obs_in = self.vec_normalize.normalize_obs(obs_in)
        action, _ = self.model.predict(obs_in, deterministic=self.deterministic)
        return int(action[0])


def evaluate_rl_policy(cfg: EnvConfig, model, vec_normalize, n_episodes=30, seed0=100):
    def make_policy(seed):
        return RLPolicyAdapter(model, vec_normalize, deterministic=True)
    return run_batch(cfg, make_policy, n_episodes=n_episodes, seed0=seed0, is_greedy=False)


if SESSION_STAGE == 'all':
    print("=== Smoke test QBER-only agent: evaluation (30 episodes, seeds 100-129) ===")
    df_qber_smoke_eval = evaluate_rl_policy(
        EnvConfig(seed=0, obs_mode="qber_budget"), model_qber_smoke, vecenv_qber_smoke,
        n_episodes=30, seed0=100
    )
    print(df_qber_smoke_eval.describe().T[["mean", "std", "min", "max"]])
    print(f"[explicit] termination breakdown:\n{df_qber_smoke_eval['termination_reason'].value_counts()}")

    print("\n=== Smoke test Messaging-aware agent: evaluation (30 episodes, seeds 100-129) ===")
    df_full_smoke_eval = evaluate_rl_policy(
        EnvConfig(seed=0, obs_mode="full"), model_full_smoke, vecenv_full_smoke,
        n_episodes=30, seed0=100
    )
    print(df_full_smoke_eval.describe().T[["mean", "std", "min", "max"]])
    print(f"[explicit] termination breakdown:\n{df_full_smoke_eval['termination_reason'].value_counts()}")

    print("\n=== Reference: baselines evaluated earlier ===")
    print(f"Random Eve            mean_reward = {df_random['total_damage_reward'].mean():.2f}")
    print(f"Greedy Eve            mean_reward = {df_greedy['total_damage_reward'].mean():.2f}")
    print(f"Best constant f=0.02  mean_reward (60 eps, from sanity_df) = "
          f"{sanity_df.loc[sanity_df['f_level']==0.02, 'mean_reward'].values[0]:.2f}")


In [ ]:
# %% Cell 17 — Calibration run: how many timesteps until convergence?
import time

if SESSION_STAGE == 'all':
    t0 = time.time()
    model_qber_cal, vecenv_qber_cal = train_ppo_agent(
        obs_mode="qber_budget", total_timesteps=200_000, seed=1,
        delay_rounds=0, save_path="qber_only_delay0_seed1_cal", verbose=0
    )
    print(f"200k-step calibration run took {time.time()-t0:.1f}s")

    df_qber_cal_eval = evaluate_rl_policy(
        EnvConfig(seed=0, obs_mode="qber_budget"), model_qber_cal, vecenv_qber_cal,
        n_episodes=50, seed0=100
    )
    print("\n=== 200k-step QBER-only agent: evaluation (50 episodes) ===")
    print(df_qber_cal_eval.describe().T[["mean", "std", "min", "max"]])
    print(f"[explicit] termination breakdown:\n{df_qber_cal_eval['termination_reason'].value_counts()}")
    print(f"\n[compare] 20k-step checkpoint mean_reward was 40.6 -- is 200k meaningfully higher?")


In [ ]:
# %% Cell 18 — Learning-curve training harness (periodic checkpoint + eval)
def train_ppo_with_learning_curve(obs_mode: str, seed: int, delay_rounds: int,
                                   total_timesteps: int, checkpoint_interval: int,
                                   eval_episodes: int = 30, eval_seed0: int = 100,
                                   ent_coef: float = 0.01, verbose_training=False):
    """
    Trains in chunks of `checkpoint_interval` steps, evaluating via the SAME
    run_batch harness used for all baselines after each chunk. Also pulls
    entropy_loss and explained_variance out of SB3's internal logger at each
    checkpoint -- these let us distinguish "converged" (entropy flatlined,
    reward flatlined) from "possibly stuck" (entropy STILL dropping while
    reward has flatlined, suggesting premature exploitation) from "still
    learning" (both still moving).
    """
    set_random_seed(seed)
    vec_env = make_training_vec_env(obs_mode, base_seed=seed, delay_rounds=delay_rounds)

    model = PPO(
        "MlpPolicy", vec_env,
        learning_rate=3e-4, n_steps=2048, batch_size=64, n_epochs=10,
        gamma=0.995, gae_lambda=0.95, ent_coef=ent_coef,
        verbose=1 if verbose_training else 0, seed=seed,
        policy_kwargs=dict(net_arch=[64, 64]),
        device=PPO_DEVICE,
    )

    n_checkpoints = total_timesteps // checkpoint_interval
    records = []
    steps_done = 0

    eval_cfg = EnvConfig(seed=0, obs_mode=obs_mode, delay_rounds=delay_rounds)

    for i in range(n_checkpoints):
        model.learn(total_timesteps=checkpoint_interval, reset_num_timesteps=False,
                    callback=LagrangeCallback(delta=EnvConfig().detection_limit))
        steps_done += checkpoint_interval

        df_eval = evaluate_rl_policy(eval_cfg, model, vec_env,
                                      n_episodes=eval_episodes, seed0=eval_seed0)

        entropy_loss = model.logger.name_to_value.get("train/entropy_loss", np.nan)
        explained_var = model.logger.name_to_value.get("train/explained_variance", np.nan)

        rec = dict(
            obs_mode=obs_mode, seed=seed, delay_rounds=delay_rounds, timesteps=steps_done,
            mean_reward=df_eval["total_damage_reward"].mean(),
            std_reward=df_eval["total_damage_reward"].std(),
            mean_episode_length=df_eval["episode_length"].mean(),
            frac_cusum_detected=df_eval["detected"].mean(),
            frac_horizon_reached=(df_eval["termination_reason"] == "horizon_reached").mean(),
            mean_max_true_qber=df_eval["max_true_qber"].mean(),
            std_max_true_qber=df_eval["max_true_qber"].std(),
            entropy_loss=entropy_loss,
            explained_variance=explained_var,
        )
        records.append(rec)
        print(f"[{obs_mode} seed={seed} delay={delay_rounds}] steps={steps_done:>7} "
              f"reward={rec['mean_reward']:7.2f} ep_len={rec['mean_episode_length']:6.1f} "
              f"entropy={entropy_loss:.3f} exp_var={explained_var:.3f} "
              f"ceiling_qber={rec['mean_max_true_qber']:.4f}±{rec['std_max_true_qber']:.4f}")

    return model, vec_env, pd.DataFrame(records)


In [ ]:
# %% Cell 18b — REVISED training helper: multi-env, no reward normalization
from stable_baselines3.common.vec_env import SubprocVecEnv

import warnings
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")

def make_training_vec_env_v2(obs_mode: str, base_seed: int, delay_rounds: int = 0,
                              n_envs: int = 8, normalize_obs=True, detection_limit=0.05, shaping_mode="none"):

    """
    REVISED AGAIN: switched from SubprocVecEnv to DummyVecEnv. SubprocVecEnv
    spawns real OS subprocesses that were never being closed between grid
    iterations, leaking memory/processes across the training loop and
    crashing the Kaggle kernel after a few runs. DummyVecEnv steps all
    n_envs environments sequentially within ONE process -- no subprocess
    spawning, no leak risk, no Kaggle sandboxing concerns. We still get the
    batch-diversity benefit (multiple independent env instances contributing
    to each PPO rollout) that fixed the v1 training instability; we just
    lose OS-level parallelism, which barely mattered anyway since this
    simulator is a fast pure-NumPy analytic model, not an expensive physics
    sim -- the earlier ~45-55s/20k-steps timing was likely dominated by
    subprocess spawn overhead, not actual computation.
    """
    def _make(rank):
        def _init():
            cfg = EnvConfig(seed=base_seed + rank, obs_mode=obs_mode, delay_rounds=delay_rounds,
                             detection_limit=detection_limit, shaping_mode=shaping_mode)
            env = EveGymEnv(cfg)
            env = Monitor(env)
            return env
        return _init

    vec_env = DummyVecEnv([_make(i) for i in range(n_envs)])
    vec_env = VecNormalize(vec_env, norm_obs=normalize_obs, norm_reward=False,
                            clip_obs=10.0, gamma=0.995)
    return vec_env


def train_ppo_with_learning_curve_v2(obs_mode: str, seed: int, delay_rounds: int,
                                      total_timesteps: int, checkpoint_interval: int,
                                      n_envs: int = 8, eval_episodes: int = 30,
                                      eval_seed0: int = 5000, ent_coef: float = 0.01,
                                      verbose_training=False):
    set_random_seed(seed)
    vec_env = make_training_vec_env_v2(obs_mode, base_seed=seed, delay_rounds=delay_rounds,
                                        n_envs=n_envs)

    model = PPO(
        "MlpPolicy", vec_env,
        learning_rate=3e-4,
        n_steps=512,              # per-env rollout length; total batch = n_steps * n_envs = 4096
        batch_size=256,
        n_epochs=10,
        gamma=0.995, gae_lambda=0.95, ent_coef=ent_coef,
        vf_coef=0.5,
        max_grad_norm=0.5,
        verbose=1 if verbose_training else 0, seed=seed,
        policy_kwargs=dict(net_arch=[64, 64]),
        device=PPO_DEVICE,
    )

    n_checkpoints = total_timesteps // checkpoint_interval
    records = []
    steps_done = 0
    eval_cfg = EnvConfig(seed=0, obs_mode=obs_mode, delay_rounds=delay_rounds)

    for i in range(n_checkpoints):
        model.learn(total_timesteps=checkpoint_interval, reset_num_timesteps=False,
                    callback=LagrangeCallback(delta=eval_cfg.detection_limit))
        steps_done += checkpoint_interval

        df_eval = evaluate_rl_policy(eval_cfg, model, vec_env,
                                      n_episodes=eval_episodes, seed0=eval_seed0)

        entropy_loss = model.logger.name_to_value.get("train/entropy_loss", np.nan)
        explained_var = model.logger.name_to_value.get("train/explained_variance", np.nan)

        rec = dict(
            obs_mode=obs_mode, seed=seed, delay_rounds=delay_rounds, timesteps=steps_done,
            mean_reward=df_eval["total_damage_reward"].mean(),
            std_reward=df_eval["total_damage_reward"].std(),
            mean_episode_length=df_eval["episode_length"].mean(),
            frac_cusum_detected=df_eval["detected"].mean(),
            frac_horizon_reached=(df_eval["termination_reason"] == "horizon_reached").mean(),
            mean_max_true_qber=df_eval["max_true_qber"].mean(),
            std_max_true_qber=df_eval["max_true_qber"].std(),
            entropy_loss=entropy_loss,
            explained_variance=explained_var,
        )
        records.append(rec)
        print(f"[{obs_mode} seed={seed} delay={delay_rounds}] steps={steps_done:>7} "
              f"reward={rec['mean_reward']:7.2f} ep_len={rec['mean_episode_length']:6.1f} "
              f"entropy={entropy_loss:.3f} exp_var={explained_var:.3f} "
              f"ceiling_qber={rec['mean_max_true_qber']:.4f}±{rec['std_max_true_qber']:.4f}")

    return model, vec_env, pd.DataFrame(records)



In [ ]:
# %% Cell 19 — Run learning curves: 2 seeds x 2 obs_modes, delay=0 (preliminary, before full grid)
if SESSION_STAGE == 'all':
    TOTAL_TIMESTEPS_CAL = 300_000
    CHECKPOINT_INTERVAL = 30_000

    learning_curve_results = []
    trained_models = {}

    for obs_mode in ["qber_budget", "full"]:
        for seed in [1, 2]:
            print(f"\n{'='*70}\nTraining: obs_mode={obs_mode}, seed={seed}, delay=0\n{'='*70}")
            model, vec_env, df_curve = train_ppo_with_learning_curve(
                obs_mode=obs_mode, seed=seed, delay_rounds=0,
                total_timesteps=TOTAL_TIMESTEPS_CAL, checkpoint_interval=CHECKPOINT_INTERVAL,
                eval_episodes=30, eval_seed0=VALIDATION_SEED0,
            )
            learning_curve_results.append(df_curve)
            trained_models[(obs_mode, seed)] = (model, vec_env)

    learning_curve_df = pd.concat(learning_curve_results, ignore_index=True)
    learning_curve_df.to_csv("learning_curves_delay0_calibration.csv", index=False)
    print("\nSaved: learning_curves_delay0_calibration.csv")


In [ ]:
# %% Cell 19b — Re-run calibration with the revised setup
if SESSION_STAGE == 'all':
    learning_curve_results_v2 = []
    trained_models_v2 = {}

    for obs_mode in ["qber_budget", "full"]:
        for seed in [1, 2]:
            print(f"\n{'='*70}\nTraining (v2): obs_mode={obs_mode}, seed={seed}, delay=0\n{'='*70}")
            model, vec_env, df_curve = train_ppo_with_learning_curve_v2(
                obs_mode=obs_mode, seed=seed, delay_rounds=0,
                total_timesteps=300_000, checkpoint_interval=30_000,
                n_envs=8, eval_episodes=30, eval_seed0=VALIDATION_SEED0,
            )
            learning_curve_results_v2.append(df_curve)
            trained_models_v2[(obs_mode, seed)] = (model, vec_env)

    learning_curve_df_v2 = pd.concat(learning_curve_results_v2, ignore_index=True)
    learning_curve_df_v2.to_csv("learning_curves_delay0_calibration_v2.csv", index=False)
    print("\nSaved: learning_curves_delay0_calibration_v2.csv")


In [ ]:
# %% Cell 20 — Plot learning curves
if SESSION_STAGE == 'all':
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    for obs_mode, color in [("qber_budget", "tab:blue"), ("full", "tab:orange")]:
        sub = learning_curve_df[learning_curve_df["obs_mode"] == obs_mode]
        for seed in sub["seed"].unique():
            s = sub[sub["seed"] == seed]
            axes[0,0].plot(s["timesteps"], s["mean_reward"], color=color, alpha=0.6,
                            marker='o', label=f"{obs_mode} seed={seed}")
            axes[0,1].plot(s["timesteps"], s["entropy_loss"], color=color, alpha=0.6, marker='o')
            axes[1,0].plot(s["timesteps"], s["explained_variance"], color=color, alpha=0.6, marker='o')
            axes[1,1].plot(s["timesteps"], s["mean_episode_length"], color=color, alpha=0.6, marker='o')

    constant_002 = float(sanity_df.loc[np.isclose(sanity_df["f_level"], 0.02), "mean_reward"].iloc[0])
    axes[0,0].axhline(constant_002, color='gray', ls='--', label='constant f=0.02 baseline')
    axes[0,0].set_title("Mean unweighted availability-damage reward"); axes[0,0].legend(fontsize=7)
    axes[0,1].set_title("Entropy loss (less negative early = more exploration; "
                         "flatlining = policy has committed)")
    axes[1,0].set_title("Explained variance (critic quality; should rise toward 1)")
    axes[1,1].set_title("Mean episode length (survival)")
    for ax in axes.flat:
        ax.set_xlabel("training timesteps"); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("learning_curves_delay0_calibration.png", dpi=120)
    plt.show()
    print("Saved: learning_curves_delay0_calibration.png")


In [ ]:
# %% Cell 20b — Plot v2 learning curves
if SESSION_STAGE == 'all':
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    for obs_mode, color in [("qber_budget", "tab:blue"), ("full", "tab:orange")]:
        sub = learning_curve_df_v2[learning_curve_df_v2["obs_mode"] == obs_mode]
        for seed in sub["seed"].unique():
            s = sub[sub["seed"] == seed]
            axes[0,0].plot(s["timesteps"], s["mean_reward"], color=color, alpha=0.6,
                            marker='o', label=f"{obs_mode} seed={seed}")
            axes[0,1].plot(s["timesteps"], s["entropy_loss"], color=color, alpha=0.6, marker='o')
            axes[1,0].plot(s["timesteps"], s["explained_variance"], color=color, alpha=0.6, marker='o')
            axes[1,1].plot(s["timesteps"], s["mean_episode_length"], color=color, alpha=0.6, marker='o')

    constant_002 = float(sanity_df.loc[np.isclose(sanity_df["f_level"], 0.02), "mean_reward"].iloc[0])
    axes[0,0].axhline(constant_002, color='gray', ls='--', label='constant f=0.02 baseline')
    axes[0,0].set_title("Mean unweighted availability-damage reward (v2)")
    axes[0,0].legend(fontsize=7)
    axes[0,1].set_title("Entropy loss")
    axes[1,0].set_title("Explained variance (should now stabilize higher)")
    axes[1,1].set_title("Mean episode length (survival)")
    for ax in axes.flat:
        ax.set_xlabel("training timesteps"); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("learning_curves_delay0_calibration_v2.png", dpi=120)
    plt.show()
    print("Saved: learning_curves_delay0_calibration_v2.png")


In [ ]:
# %% Cell 21 — Continue training the v2 models further (600k total), same checkpoint cadence
def continue_training_with_learning_curve(model, vec_env, obs_mode, seed, delay_rounds,
                                           additional_timesteps, checkpoint_interval,
                                           steps_already_done, eval_episodes=30, eval_seed0=VALIDATION_SEED0):
    records = []
    steps_done = steps_already_done
    eval_cfg = EnvConfig(seed=0, obs_mode=obs_mode, delay_rounds=delay_rounds)
    n_checkpoints = additional_timesteps // checkpoint_interval

    for i in range(n_checkpoints):
        model.learn(total_timesteps=checkpoint_interval, reset_num_timesteps=False,
                    callback=LagrangeCallback(delta=EnvConfig().detection_limit))
        steps_done += checkpoint_interval

        df_eval = evaluate_rl_policy(eval_cfg, model, vec_env,
                                      n_episodes=eval_episodes, seed0=eval_seed0)

        entropy_loss = model.logger.name_to_value.get("train/entropy_loss", np.nan)
        explained_var = model.logger.name_to_value.get("train/explained_variance", np.nan)
        value_loss = model.logger.name_to_value.get("train/value_loss", np.nan)

        rec = dict(
            obs_mode=obs_mode, seed=seed, delay_rounds=delay_rounds, timesteps=steps_done,
            mean_reward=df_eval["total_damage_reward"].mean(),
            std_reward=df_eval["total_damage_reward"].std(),
            mean_episode_length=df_eval["episode_length"].mean(),
            frac_cusum_detected=df_eval["detected"].mean(),
            frac_horizon_reached=(df_eval["termination_reason"] == "horizon_reached").mean(),
            entropy_loss=entropy_loss, explained_variance=explained_var, value_loss=value_loss,
        )
        records.append(rec)
        print(f"[{obs_mode} seed={seed} delay={delay_rounds}] steps={steps_done:>7} "
              f"reward={rec['mean_reward']:7.2f} ep_len={rec['mean_episode_length']:6.1f} "
              f"entropy={entropy_loss:.3f} exp_var={explained_var:.3f} vf_loss={value_loss:.4f}")

    return pd.DataFrame(records)


if SESSION_STAGE == 'all':
    continued_results = []
    for (obs_mode, seed), (model, vec_env) in trained_models_v2.items():
        print(f"\n{'='*70}\nContinuing: obs_mode={obs_mode}, seed={seed}\n{'='*70}")
        df_cont = continue_training_with_learning_curve(
            model, vec_env, obs_mode=obs_mode, seed=seed, delay_rounds=0,
            additional_timesteps=300_000, checkpoint_interval=30_000,
            steps_already_done=300_000, eval_episodes=30, eval_seed0=VALIDATION_SEED0,
        )
        continued_results.append(df_cont)

    learning_curve_df_v2_extended = pd.concat(
        [learning_curve_df_v2] + continued_results, ignore_index=True
    )
    learning_curve_df_v2_extended.to_csv("learning_curves_delay0_calibration_v2_extended.csv", index=False)
    print("\nSaved: learning_curves_delay0_calibration_v2_extended.csv")


In [ ]:
# %% Cell 21b — Convergence check at delay=50 before committing the full grid budget
if SESSION_STAGE == 'all':
    model_d50, vecenv_d50, curve_d50 = train_ppo_with_learning_curve_v2(
        obs_mode="full", seed=1, delay_rounds=50,
        total_timesteps=600_000, checkpoint_interval=30_000,
        n_envs=8, eval_episodes=30, eval_seed0=VALIDATION_SEED0,
    )
    curve_d50.to_csv("learning_curve_delay50_seed1_full.csv", index=False)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].plot(curve_d50["timesteps"], curve_d50["mean_reward"], marker="o")
    ax[0].set_title("delay=50: mean reward vs timesteps"); ax[0].grid(alpha=0.3)
    ax[1].plot(curve_d50["timesteps"], curve_d50["explained_variance"], marker="o", label="explained_variance")
    ax[1].plot(curve_d50["timesteps"], -curve_d50["entropy_loss"], marker="o", label="-entropy_loss")
    ax[1].axvline(400_000, color="red", ls="--", label="planned grid budget")
    ax[1].legend(); ax[1].grid(alpha=0.3); ax[1].set_title("Convergence diagnostics")
    plt.tight_layout(); plt.savefig("convergence_check_delay50.png", dpi=120); plt.show()

    print("[decision] if reward/explained_variance are still rising at the red line, "
          "bump GRID_TIMESTEPS in Cell 22, or use a delay-dependent budget.")


In [ ]:
# %% Cell 22 — Full experimental grid: setup and constants
import os
import json
import time
from scipy import stats as scipy_stats
from scipy.stats import beta as beta_dist

# --- QUICK-TEST TOGGLE ---
# Set True first to validate the entire pipeline (dirs, resumability, CSV
# writing, plotting) end-to-end in a few minutes with tiny budgets. Once
# confirmed working, set False and rerun for the real experiment.
QUICK_TEST = False

if QUICK_TEST:
    GRID_TIMESTEPS = 20_000
    GRID_SEEDS = [1, 2]
    GRID_EVAL_EPISODES = 15
    print("[QUICK_TEST=True] Using tiny budgets to validate pipeline plumbing only. "
          "Results from this pass are NOT meaningful -- rerun with QUICK_TEST=False for real results.")
else:
    GRID_TIMESTEPS = 400_000
    GRID_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]   # 10 seeds for a defensible headline claim
    GRID_EVAL_EPISODES = 200
    print("[QUICK_TEST=False] Full experimental budget -- this will take a long time. See wall-clock estimate.")

# Delay conditions, justified from Phase-1 validated trace: the natural
# pool build-up/drawdown cycle observed under the validated dynamics spanned
# roughly 40-50 rounds (calm accumulation) followed by a comparable drain
# during HIGH-value bursts. We pick:
#   0  -> no buffering (attack-time == consumption-time; near-bandit regime)
#   15 -> moderate buffering (~1/3 of the natural cycle length)
#   50 -> large buffering (~matches the natural full cycle length)
DELAY_CONDITIONS = [0, 15, 50]
OBS_MODES = ["qber_budget", "qber_pool", "metadata_budget", "detector_oracle", "full"]

CHECKPOINT_INTERVAL = 40_000 if not QUICK_TEST else 10_000
N_ENVS = 8
BASELINE_VALIDATION_SEED0 = 5000
BASELINE_TEST_SEEDS = [100000, 102000, 104000, 106000, 108000]

MODEL_DIR = os.path.join("models", "availability_damage_v2", "quicktest" if QUICK_TEST else "final")
RESULTS_DIR = os.path.join("results", "availability_damage_v2", "quicktest" if QUICK_TEST else "final")

# This is a fresh one-shot run: existing Kaggle output folders are not imported.
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

RL_RESULTS_CSV = os.path.join(RESULTS_DIR, f"rl_grid_results{'_quicktest' if QUICK_TEST else ''}.csv")
BASELINE_RESULTS_CSV = os.path.join(RESULTS_DIR, f"baseline_grid_results{'_quicktest' if QUICK_TEST else ''}.csv")

print(f"Grid: {len(OBS_MODES)} obs_modes x {len(DELAY_CONDITIONS)} delays x {len(GRID_SEEDS)} seeds "
      f"= {len(OBS_MODES)*len(DELAY_CONDITIONS)*len(GRID_SEEDS)} training runs, "
      f"{GRID_TIMESTEPS} steps each")


def csv_contains_all_keys(path, columns, expected_keys):
    """True when the final results CSV contains every planned result key."""
    if not os.path.exists(path):
        return False
    frame = pd.read_csv(path)
    if not set(columns).issubset(frame.columns):
        return False
    observed = set()
    for values in frame[columns].itertuples(index=False, name=None):
        normalized = tuple(float(x) if isinstance(x, (float, np.floating)) else x for x in values)
        observed.add(normalized)
    expected = {tuple(float(x) if isinstance(x, (float, np.floating)) else x for x in key)
                for key in expected_keys}
    return expected.issubset(observed)


In [ ]:
# %% Cell 23 — Fresh PPO training helper
def model_paths(obs_mode, delay, seed, detection_limit=0.05):
    tag = f"{obs_mode}_delay{delay}_delta{detection_limit:g}_seed{seed}{'_quicktest' if QUICK_TEST else ''}"
    return (os.path.join(MODEL_DIR, f"{tag}.zip"),
            os.path.join(MODEL_DIR, f"{tag}_vecnorm.pkl"),
            os.path.join(MODEL_DIR, f"{tag}_meta.json"))


def train_agent(obs_mode, delay, seed, total_timesteps, checkpoint_interval,
                n_envs=N_ENVS, detection_limit=0.05):
    """
    Trains a fresh PPO agent for every requested combination. Existing model
    files are overwritten and are never used to skip training.
    """
    model_path, vecnorm_path, meta_path = model_paths(obs_mode, delay, seed, detection_limit)

    print(f"[TRAIN] obs_mode={obs_mode} delay={delay} seed={seed} -> {total_timesteps} steps")
    t0 = time.time()
    set_random_seed(seed)
    vec_env = make_training_vec_env_v2(obs_mode, base_seed=seed, delay_rounds=delay, n_envs=n_envs,
        detection_limit=detection_limit)

    model = PPO(
        "MlpPolicy", vec_env,
        learning_rate=3e-4, n_steps=512, batch_size=256, n_epochs=10,
        gamma=0.999, gae_lambda=0.95, ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5,
        verbose=0, seed=seed, policy_kwargs=dict(net_arch=[64, 64]),
        device=PPO_DEVICE,
    )

    n_checkpoints = max(1, total_timesteps // checkpoint_interval)
    learning_curve = []
    eval_cfg = EnvConfig(seed=0, obs_mode=obs_mode, delay_rounds=delay,
                         detection_limit=detection_limit)
    lagrange_callback = LagrangeCallback(delta=eval_cfg.detection_limit)
    steps_done = 0

    def _check_action_collapse(model, vec_env, cfg, n_episodes=10, seed0=VALIDATION_SEED0):
        counts = np.zeros(len(cfg.intercept_levels))
        for i in range(n_episodes):
            env = EveEnv(cfg)
            obs = env.reset(seed=seed0 + i)
            done = False
            while not done:
                obs_in = vec_env.normalize_obs(obs.reshape(1, -1).astype(np.float32))
                a, _ = model.predict(obs_in, deterministic=True)
                counts[int(a[0])] += 1
                obs, r, done, info = env.step(int(a[0]))
        total = counts.sum()
        return counts / total if total > 0 else counts

    for i in range(n_checkpoints):
        model.learn(total_timesteps=checkpoint_interval, reset_num_timesteps=False,
                    callback=lagrange_callback)
        steps_done += checkpoint_interval
        df_eval = evaluate_rl_policy(eval_cfg, model, vec_env, n_episodes=100, seed0=VALIDATION_SEED0)

        entropy_loss = model.logger.name_to_value.get("train/entropy_loss", np.nan)
        explained_var = model.logger.name_to_value.get("train/explained_variance", np.nan)
        value_loss = model.logger.name_to_value.get("train/value_loss", np.nan)
        action_dist = _check_action_collapse(model, vec_env, eval_cfg, n_episodes=10)
        dominant_frac = float(action_dist.max())

        learning_curve.append(dict(
            timesteps=steps_done,
            mean_reward=float(df_eval["total_unweighted_damage_reward"].mean()),
            std_reward=float(df_eval["total_damage_reward"].std()),
            frac_cusum_detected=float(df_eval["detected"].mean()),
            entropy_loss=None if np.isnan(entropy_loss) else float(entropy_loss),
            explained_variance=None if np.isnan(explained_var) else float(explained_var),
            value_loss=None if np.isnan(value_loss) else float(value_loss),
            dominant_action_frac=dominant_frac,
            lagrange_lambda=lagrange_callback.lmbda,
            lagrange_trajectory=lagrange_callback.trajectory.copy(),
        ))
        if i == n_checkpoints - 1 and dominant_frac > 0.95:
            print(f"  [WARNING] policy may have collapsed: {dominant_frac:.2f} mass on one action "
                  f"(dist={np.round(action_dist,3)})")

    elapsed = time.time() - t0
    print(f"  done in {elapsed:.0f}s, final reward={learning_curve[-1]['mean_reward']:.2f}, "
          f"dominant_action_frac={learning_curve[-1]['dominant_action_frac']:.2f}")

    model.save(model_path)
    vec_env.save(vecnorm_path)
    with open(meta_path, "w") as f:
        json.dump(dict(obs_mode=obs_mode, delay=delay, seed=seed,
                        total_timesteps=total_timesteps, elapsed_seconds=elapsed, device=PPO_DEVICE,
                        detection_limit=eval_cfg.detection_limit,
                        lagrange_trajectory=lagrange_callback.trajectory,
                        training_feasibility=(np.mean([x["detection_rate"] <= eval_cfg.detection_limit
                            for x in lagrange_callback.trajectory]) if lagrange_callback.trajectory else None),
                        learning_curve=learning_curve), f, indent=2)

    return model, vec_env, learning_curve


In [ ]:
# %% Cell 24 — Full RL grid (fresh training)
import gc

grid_start_time = time.time()
grid_final_results = []
all_grid_runs = [(obs_mode, delay, seed)
    for obs_mode in OBS_MODES for delay in DELAY_CONDITIONS for seed in GRID_SEEDS]
print(f"[grid] Training all {len(all_grid_runs)} combinations from scratch.")
for run_number, (obs_mode, delay, seed) in enumerate(all_grid_runs, start=1):
    print(f"\n[grid {run_number}/{len(all_grid_runs)}] obs_mode={obs_mode}, delay={delay}, seed={seed}")
    model, vec_env, curve = train_agent(
        obs_mode, delay, seed, GRID_TIMESTEPS, CHECKPOINT_INTERVAL, n_envs=N_ENVS)
    eval_cfg = EnvConfig(seed=0, obs_mode=obs_mode, delay_rounds=delay)
    df_eval = evaluate_rl_policy(eval_cfg, model, vec_env,
                                 n_episodes=GRID_EVAL_EPISODES, seed0=TEST_SEED0)
    detected_n = int(df_eval["detected"].sum())
    n_eval = len(df_eval)
    detection_ci_lo = 0.0 if detected_n == 0 else float(
        beta_dist.ppf(0.025, detected_n, n_eval-detected_n+1))
    detection_ci_hi = 1.0 if detected_n == n_eval else float(
        beta_dist.ppf(0.975, detected_n+1, n_eval-detected_n))
    _, _, meta_path = model_paths(obs_mode, delay, seed)
    if os.path.exists(meta_path):
        with open(meta_path) as meta_file:
            run_meta = json.load(meta_file)
        run_meta.update(test_detection_rate=float(df_eval["detected"].mean()),
            test_detection_ci=[detection_ci_lo,detection_ci_hi],
            test_feasible=bool(df_eval["detected"].mean() <= EnvConfig().detection_limit),
            test_episodes=n_eval)
        with open(meta_path,"w") as meta_file:
            json.dump(run_meta,meta_file,indent=2)
    grid_final_results.append(dict(
        agent="RL", obs_mode=obs_mode, delay=delay, seed=seed,
        mean_reward=float(df_eval["total_unweighted_damage_reward"].mean()),
        std_reward=float(df_eval["total_damage_reward"].std()),
        mean_episode_length=float(df_eval["episode_length"].mean()),
        frac_cusum_detected=float(df_eval["detected"].mean()),
        detection_rate=float(df_eval["detected"].mean()),
        detection_ci_lo=detection_ci_lo, detection_ci_hi=detection_ci_hi,
        frac_horizon_reached=float((df_eval["termination_reason"] == "horizon_reached").mean()),
        frac_hard_abort=float((df_eval["termination_reason"] == "hard_abort").mean()),
        n_eval_episodes=n_eval, training_device=PPO_DEVICE))
    vec_env.close()
    del model, vec_env
    gc.collect()

rl_grid_df = pd.DataFrame(grid_final_results)
rl_grid_df.to_csv(RL_RESULTS_CSV,index=False)
GRID_STAGE_COMPLETE = (len(rl_grid_df) == len(all_grid_runs))
print(f"[grid] All {len(rl_grid_df)} runs complete in {(time.time()-grid_start_time)/3600:.2f} hours.")
print(rl_grid_df.groupby(["obs_mode","delay"])["mean_reward"].agg(["mean","std","count"]))


In [ ]:
# %% Cell 24b — Policy collapse / degeneracy diagnostic
def action_histogram(model_path, vecnorm_path, cfg, n_episodes=10, seed0=100):
    vec_env = make_training_vec_env_v2(cfg.obs_mode, base_seed=0, delay_rounds=cfg.delay_rounds, n_envs=1)
    vec_env = VecNormalize.load(vecnorm_path, vec_env)
    vec_env.training = False
    model = PPO.load(model_path, device=PPO_DEVICE)

    counts = np.zeros(len(cfg.intercept_levels))
    for i in range(n_episodes):
        env = EveEnv(cfg)
        obs = env.reset(seed=seed0 + i)
        done = False
        while not done:
            obs_in = obs.reshape(1, -1).astype(np.float32)
            obs_in = vec_env.normalize_obs(obs_in)
            a, _ = model.predict(obs_in, deterministic=True)
            counts[int(a[0])] += 1
            obs, r, done, info = env.step(int(a[0]))
    vec_env.close()
    return counts / counts.sum()

if SESSION_STAGE == 'analysis' or (SESSION_STAGE == 'all' and GRID_STAGE_COMPLETE):
    print("=== Action distribution check: delay=50 models ===")
    for obs_mode in ["qber_budget", "full"]:
        for seed in GRID_SEEDS:
            model_path, vecnorm_path, _ = model_paths(obs_mode, 50, seed)
            cfg_diag = EnvConfig(seed=0, obs_mode=obs_mode, delay_rounds=50)
            dist = action_histogram(model_path, vecnorm_path, cfg_diag, n_episodes=15, seed0=100)
            flag = "  <-- COLLAPSED" if dist.max() > 0.90 else ""
            print(f"{obs_mode:10s} seed={seed}: {np.round(dist,3)}  (dominant={dist.max():.2f}){flag}")
else:
    print("[diagnostic] Waiting for the complete RL grid.")


In [ ]:
# %% Cell 24c — Full Lagrangian constrained-PPO risk sweep
DETECTION_TARGETS = [0.01,0.05,0.10]
RISK_SWEEP_SEEDS = GRID_SEEDS[:3] if QUICK_TEST else GRID_SEEDS
RISK_SWEEP_STEPS = GRID_TIMESTEPS
PARETO_DELAYS = DELAY_CONDITIONS
RISK_SWEEP_EVAL_EPISODES = min(GRID_EVAL_EPISODES,200) if QUICK_TEST else GRID_EVAL_EPISODES
RISK_SWEEP_CSV = os.path.join(RESULTS_DIR,f"risk_sweep{'_quicktest' if QUICK_TEST else ''}.csv")
risk_rows = []
all_risk_runs = [(obs_mode,delay,float(delta),seed) for obs_mode in OBS_MODES
    for delay in PARETO_DELAYS for delta in DETECTION_TARGETS for seed in RISK_SWEEP_SEEDS]
print(f"[risk] Training all {len(all_risk_runs)} constrained policies from scratch.")
for run_number,(obs_mode,delay,delta,seed) in enumerate(all_risk_runs,start=1):
    print(f"[risk {run_number}/{len(all_risk_runs)}] {obs_mode=}, {delay=}, {delta=}, {seed=}")
    set_random_seed(seed)
    vec_env=make_training_vec_env_v2(obs_mode,seed,delay_rounds=delay,
        n_envs=N_ENVS,detection_limit=delta)
    model=PPO("MlpPolicy",vec_env,learning_rate=3e-4,n_steps=512,
        batch_size=256,n_epochs=10,gamma=0.999,gae_lambda=0.95,
        ent_coef=0.01,vf_coef=0.5,max_grad_norm=0.5,verbose=0,seed=seed,
        policy_kwargs=dict(net_arch=[64,64]),device=PPO_DEVICE)
    lagrange_callback=LagrangeCallback(delta=delta)
    model.learn(total_timesteps=RISK_SWEEP_STEPS,callback=lagrange_callback)
    val_cfg=EnvConfig(seed=0,obs_mode=obs_mode,delay_rounds=delay,detection_limit=delta)
    evaluate_rl_policy(val_cfg,model,vec_env,n_episodes=100,seed0=VALIDATION_SEED0)
    test=evaluate_rl_policy(val_cfg,model,vec_env,
        n_episodes=RISK_SWEEP_EVAL_EPISODES,seed0=TEST_SEED0)
    detected=float(test["detected"].mean())
    risk_rows.append(dict(obs_mode=obs_mode,delay=delay,delta=delta,seed=seed,
        mean_reward=float(test["total_damage_reward"].mean()),detection_rate=detected,
        feasible=detected<=delta,final_lambda=lagrange_callback.lmbda,
        training_device=PPO_DEVICE))
    vec_env.close(); del model,vec_env; gc.collect()
pareto_df=pd.DataFrame(risk_rows)
pareto_df.to_csv(RISK_SWEEP_CSV,index=False)
RISK_STAGE_COMPLETE=(len(pareto_df)==len(all_risk_runs))
print(pareto_df.groupby(["obs_mode","delay","delta"])[
    ["mean_reward","detection_rate","feasible"]].mean())


In [ ]:
# %% Cell 25 — Full paired baseline evaluation
BASELINE_VALIDATION_EPISODES = 200 if not QUICK_TEST else 30
baseline_rows = []
for delay in DELAY_CONDITIONS:
    cfg_delay=EnvConfig(seed=0,obs_mode="full",delay_rounds=delay)
    validation_seed0=BASELINE_VALIDATION_SEED0+1000*delay
    test_seed0=TEST_SEED0+1000*delay
    calib_env=EveEnv(cfg_delay); calib_env.reset(seed=777)
    rate_samples=[]
    for _ in range(cfg_delay.max_rounds):
        _,_,done,_=calib_env.step(0)
        rate_samples.append(dict(zip(calib_env.feature_names,calib_env.frame_buf[-1]))["traffic_rate_obs"])
        if done: break
    greedy=GreedyTrafficEvePolicy(cfg_delay.intercept_levels,cfg_delay.qber_abort)
    greedy.calibrate(np.asarray(rate_samples))
    const_scores=[]
    for action_idx,f_level in enumerate(cfg_delay.intercept_levels):
        val=run_batch(cfg_delay,lambda seed,a=action_idx: ConstantActionPolicy(a),
            n_episodes=BASELINE_VALIDATION_EPISODES,seed0=validation_seed0)
        const_scores.append((action_idx,f_level,float(val["total_damage_reward"].mean())))
    best_idx,best_f,_=max(const_scores,key=lambda row:row[2])
    highest_idx=len(cfg_delay.intercept_levels)-1
    emission_mean,emission_std=fit_metadata_emissions(cfg_delay)
    risk_curve=(arl_df["f"].to_numpy(),arl_df["frac_cusum_terminated"].to_numpy())
    qmdp_candidates=[]
    for penalty in (0.1,0.5,1.0,2.0,5.0):
        q_values=build_qmdp_values(cfg_delay,risk_curve,risk_penalty=penalty)
        factory=lambda seed,q=q_values: ApproximateQMDPPolicy(cfg_delay,emission_mean,emission_std,q)
        val=run_batch(cfg_delay,factory,BASELINE_VALIDATION_EPISODES,seed0=validation_seed0)
        risk=float(val["detected"].mean())
        qmdp_candidates.append((risk,float(val["total_damage_reward"].mean()),penalty,q_values))
    feasible_qmdp=[x for x in qmdp_candidates if x[0]<=cfg_delay.detection_limit]
    selected_qmdp=max(feasible_qmdp,key=lambda x:x[1]) if feasible_qmdp else min(qmdp_candidates,key=lambda x:x[0])
    qmdp_factory=lambda seed,q=selected_qmdp[3]: ApproximateQMDPPolicy(cfg_delay,emission_mean,emission_std,q)
    policy_factories={
        "Random":lambda seed: RandomEvePolicy(len(cfg_delay.intercept_levels),np.random.default_rng(seed)),
        "Greedy":lambda seed: greedy,
        "BudgetMatched":lambda seed: BudgetMatchedPolicy(cfg_delay.intercept_levels,cfg_delay.total_budget_qubits,
            cfg_delay.max_rounds,cfg_delay.n_qubits_per_round),
        "DetectorAware":lambda seed: DetectorAwarePolicy(cfg_delay.intercept_levels,cfg_delay.qber_baseline,
            cfg_delay.cusum_slack,cfg_delay.cusum_threshold),
        "TimeOnly":lambda seed: TimeOnlyPolicy(cfg_delay.max_rounds,highest_idx),
        "StateOracle":lambda seed,q=selected_qmdp[3]: StateOraclePolicy(highest_idx,q_values=q,
            budget_step=min(f for f in cfg_delay.intercept_levels if f>0)),
        "HMMMetadata":lambda seed: MetadataHMMPolicy(cfg_delay,emission_mean,emission_std),
        "ApproxQMDP":qmdp_factory,
        f"BestConstant(f={best_f:g})":lambda seed,a=best_idx: ConstantActionPolicy(a),
    }
    for agent,factory in policy_factories.items():
        df=run_batch(cfg_delay,factory,n_episodes=GRID_EVAL_EPISODES,
                     seed0=test_seed0,is_greedy=(agent=="Greedy"))
        df["agent"],df["delay"]=agent,delay
        df["episode_seed"]=np.arange(test_seed0,test_seed0+len(df))
        baseline_rows.append(df)
baseline_df=pd.concat(baseline_rows,ignore_index=True)
baseline_df.to_csv(BASELINE_RESULTS_CSV,index=False)
BASELINES_STAGE_COMPLETE=(baseline_df["delay"].nunique()==len(DELAY_CONDITIONS))
print(baseline_df.groupby(["agent","delay"])[
    ["total_damage_reward","detected","total_availability_damage"]].agg(["mean","std","count"]))


In [ ]:
# %% Cell 26 — Seed-level inference for RL ablations (episode rows are not independent training replicates)

def bootstrap_mean_ci(values, confidence=0.95, draws=10000, seed=321):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not len(values): return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = rng.choice(values, size=(draws, len(values)), replace=True).mean(axis=1)
    alpha = 1.0-confidence
    return float(values.mean()), float(np.quantile(boot, alpha/2)), float(np.quantile(boot, 1-alpha/2))


def paired_bootstrap_ci(differences, confidence=0.95, draws=10000, seed=321):
    return bootstrap_mean_ci(differences, confidence=confidence, draws=draws, seed=seed)


def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    if not len(x) or not len(y): return np.nan
    # Pairwise stochastic dominance, normalized to [-1, 1].
    return float((np.sum(x[:, None] > y[None, :]) - np.sum(x[:, None] < y[None, :])) / (len(x)*len(y)))

GRID_STAGE_COMPLETE = csv_contains_all_keys(RL_RESULTS_CSV, ["obs_mode","delay","seed"],
    [(o,d,seed) for o in OBS_MODES for d in DELAY_CONDITIONS for seed in GRID_SEEDS])
BASELINES_STAGE_COMPLETE = csv_contains_all_keys(BASELINE_RESULTS_CSV, ["delay"],
    [(d,) for d in DELAY_CONDITIONS])
ANALYSIS_READY = (SESSION_STAGE == "analysis" or
    (SESSION_STAGE == "all" and GRID_STAGE_COMPLETE and BASELINES_STAGE_COMPLETE))
if ANALYSIS_READY:
    if not os.path.exists(RL_RESULTS_CSV) or not os.path.exists(BASELINE_RESULTS_CSV):
        raise FileNotFoundError("Analysis needs completed RL grid and baseline CSVs.")
    rl_grid_df = pd.read_csv(RL_RESULTS_CSV)
    baseline_df = pd.read_csv(BASELINE_RESULTS_CSV)
    risk_path = os.path.join(RESULTS_DIR, f"risk_sweep{'_quicktest' if QUICK_TEST else ''}.csv")
    pareto_df = pd.read_csv(risk_path) if os.path.exists(risk_path) else pd.DataFrame()

if ANALYSIS_READY:
    summary_rows = []
    for obs_mode in OBS_MODES:
        for delay in DELAY_CONDITIONS:
            sub = rl_grid_df[(rl_grid_df.obs_mode == obs_mode) & (rl_grid_df.delay == delay)]
            vals = sub["mean_reward"].to_numpy(dtype=float)
            mean, lo, hi = bootstrap_mean_ci(vals, seed=321+delay)
            summary_rows.append(dict(agent=obs_mode, delay=delay, mean_reward=mean,
                ci_lo=lo, ci_hi=hi, n_training_seeds=len(vals), resampling_unit="training_seed"))
    for agent in baseline_df["agent"].unique():
        for delay in DELAY_CONDITIONS:
            vals = baseline_df[(baseline_df.agent == agent) & (baseline_df.delay == delay)]["total_damage_reward"].values
            mean, lo, hi = bootstrap_mean_ci(vals, seed=654+delay)
            summary_rows.append(dict(agent=agent, delay=delay, mean_reward=mean,
                ci_lo=lo, ci_hi=hi, n_episodes=len(vals), resampling_unit="paired_episode"))
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(RESULTS_DIR, f"summary_with_ci{'_quicktest' if QUICK_TEST else ''}.csv"), index=False)
    print(summary_df.to_string(index=False))

    # Primary contrasts: metadata value at fixed QBER/budget input, then pool observability
    # conditional on metadata. Seeds are matched by seed ID for the bootstrap difference.
    contrast_specs = [("metadata_budget", "qber_budget", "metadata_gain"),
                      ("full", "metadata_budget", "pool_gain")]
    test_rows = []
    for delay in DELAY_CONDITIONS:
        for better_mode, reference_mode, contrast in contrast_specs:
            a = rl_grid_df[(rl_grid_df.obs_mode == better_mode) & (rl_grid_df.delay == delay)][["seed", "mean_reward"]]
            b = rl_grid_df[(rl_grid_df.obs_mode == reference_mode) & (rl_grid_df.delay == delay)][["seed", "mean_reward"]]
            paired = a.merge(b, on="seed", suffixes=("_a", "_b")).dropna()
            x, y = paired["mean_reward_a"].to_numpy(), paired["mean_reward_b"].to_numpy()
            if len(x) >= 2:
                diff = x-y
                mean, lo, hi = paired_bootstrap_ci(diff, seed=900+delay)
                mw = scipy_stats.mannwhitneyu(x, y, alternative="two-sided", method="auto")
                p_value = float(mw.pvalue)
            else:
                mean = lo = hi = p_value = np.nan
            test_rows.append(dict(delay=delay, contrast=contrast, better_mode=better_mode,
                reference_mode=reference_mode, mean_paired_difference=mean, ci_lo=lo, ci_hi=hi,
                n_training_seeds=len(x), mann_whitney_u=(float(mw.statistic) if len(x)>=2 else np.nan),
                p_value=p_value, cliffs_delta=cliffs_delta(x,y)))

    # Holm-Bonferroni adjustment across all planned ablation tests, ignoring unavailable p-values.
    valid = [(i, row["p_value"]) for i,row in enumerate(test_rows) if np.isfinite(row["p_value"])]
    ordered = sorted(valid, key=lambda item:item[1])
    prev = 0.0
    m = len(ordered)
    for rank,(row_i,pv) in enumerate(ordered):
        adjusted = min(1.0, (m-rank)*pv)
        prev = max(prev, adjusted)
        test_rows[row_i]["p_holm_adjusted"] = prev
    for row in test_rows:
        row.setdefault("p_holm_adjusted", np.nan)

    ablation_tests_df = pd.DataFrame(test_rows)
    ablation_tests_df.to_csv(os.path.join(RESULTS_DIR, "ablation_seed_statistics.csv"), index=False)
    print("\n=== Seed-level ablation tests (Mann–Whitney U; paired-seed bootstrap CIs) ===")
    print(ablation_tests_df.to_string(index=False))
    if any(ablation_tests_df["n_training_seeds"] < 2):
        print("[warning] fewer than two completed training seeds in at least one contrast; inference is not available.")
else:
    print("[analysis] Waiting for completed grid and baseline results.")


In [ ]:
# %% Cell 26b — Matched-risk comparison via Pareto-frontier interpolation
TARGET_DETECTION_RISK = 0.10

def interpolate_at_risk(df, target_risk):
    d = df.groupby("detection_rate")["mean_reward"].mean().reset_index().sort_values("detection_rate")
    x, y = d["detection_rate"].values, d["mean_reward"].values
    if len(x) < 2 or target_risk < x.min() or target_risk > x.max():
        return np.nan
    return float(np.interp(target_risk, x, y))

RISK_STAGE_COMPLETE = csv_contains_all_keys(RISK_SWEEP_CSV,
    ["obs_mode","delay","delta","seed"],
    [(o,d,float(delta),seed) for o in OBS_MODES for d in PARETO_DELAYS
     for delta in DETECTION_TARGETS for seed in RISK_SWEEP_SEEDS])
if ANALYSIS_READY and RISK_STAGE_COMPLETE and not pareto_df.empty:
    matched_rows = []
    for obs_mode in OBS_MODES:
        for delay in DELAY_CONDITIONS:
            sub = pareto_df[(pareto_df.obs_mode == obs_mode) & (pareto_df.delay == delay)]
            matched_rows.append(dict(obs_mode=obs_mode, delay=delay,
                                      reward_at_matched_risk=interpolate_at_risk(sub, TARGET_DETECTION_RISK)))

    matched_df = pd.DataFrame(matched_rows)
    coverage = matched_df["reward_at_matched_risk"].notna().all()
    if not coverage:
        print("[coverage warning] target detection risk is outside at least one achieved Pareto range; values are NaN.")
    print(matched_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 5))
    for obs_mode, color in [("qber_pool", "tab:blue"), ("full", "tab:orange")]:
        sub = matched_df[matched_df.obs_mode == obs_mode].sort_values("delay")
        ax.plot(sub["delay"], sub["reward_at_matched_risk"], marker="o", label=obs_mode, color=color)
    ax.set_xlabel("buffering delay"); ax.set_ylabel(f"reward @ {TARGET_DETECTION_RISK:.0%} detection risk")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, f"matched_risk_comparison{'_quicktest' if QUICK_TEST else ''}.png"), dpi=130); plt.show()
else:
    print("[analysis] Matched-risk plot will run when the risk sweep is complete.")


In [ ]:
# %% Cell 27 — Ablation plot: metadata value and pool-observation value by delay
if ANALYSIS_READY:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    agents_to_plot = list(OBS_MODES) + ["Random", "Greedy", "BudgetMatched", "DetectorAware",
        "TimeOnly", "StateOracle", "HMMMetadata", "ApproxQMDP"]
    agents_to_plot += [a for a in summary_df["agent"].unique() if a.startswith("BestConstant")]
    colors = plt.cm.tab20(np.linspace(0, 1, len(agents_to_plot)))
    width = 0.8 / len(agents_to_plot)
    x = np.arange(len(DELAY_CONDITIONS))
    for i, agent in enumerate(agents_to_plot):
        sub = summary_df[summary_df.agent == agent].set_index("delay").reindex(DELAY_CONDITIONS)
        means = sub["mean_reward"].values
        errs = np.abs(sub[["ci_lo", "ci_hi"]].values.T - means)
        errs = np.nan_to_num(errs)
        axes[0].bar(x + i*width - 0.4 + width/2, means, width, yerr=errs, capsize=2,
                    label=agent, color=colors[i])
    axes[0].set_xticks(x); axes[0].set_xticklabels([str(d) for d in DELAY_CONDITIONS])
    axes[0].set_xlabel("buffering delay (rounds)")
    axes[0].set_ylabel("mean unweighted availability-damage reward")
    axes[0].set_title("Agent reward by observation mode")
    axes[0].legend(fontsize=6, ncol=2)
    axes[0].axhline(0, color="black", lw=0.5)

    for contrast, label, color in [("metadata_gain", "metadata_budget − qber_budget", "darkred"),
                                   ("pool_gain", "full − metadata_budget", "darkblue")]:
        sub = ablation_tests_df[ablation_tests_df.contrast == contrast].sort_values("delay")
        axes[1].errorbar(sub["delay"], sub["mean_paired_difference"],
            yerr=[sub["mean_paired_difference"]-sub["ci_lo"], sub["ci_hi"]-sub["mean_paired_difference"]],
            marker="o", capsize=4, linewidth=2, label=label, color=color)
    axes[1].axhline(0, color="black", ls="--", lw=1)
    axes[1].set_xlabel("buffering delay (rounds)")
    axes[1].set_ylabel("paired mean reward difference across training seeds")
    axes[1].set_title("Separate traffic-metadata and pool observability effects")
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"observation_ablation{'_quicktest' if QUICK_TEST else ''}.png"), dpi=130)
    plt.show()
else:
    print("[analysis] Waiting for completed grid and baseline results.")


In [ ]:
# %% Cell 27b — Statistical tests moved to Cell 26: Mann–Whitney U, paired-seed bootstrap CI, Cliff delta, Holm correction.
if ANALYSIS_READY:
    print("Seed-level ablation inference is available in ablation_tests_df and ablation_seed_statistics.csv.")
else:
    print("[analysis] Waiting for completed grid and baseline results.")


In [ ]:
# %% Cell 28 — Full adaptive evaluation across traffic-shaping defenses
SHAPING_MODES=["none","constant_rate","padding","cover_traffic","jitter"]
BEST_DELAY_FOR_DEFENSE_TEST=15
DEFENSE_CSV=os.path.join(RESULTS_DIR,f"defense_sweep{'_quicktest' if QUICK_TEST else ''}.csv")
defense_rows=[]
all_defense_runs=[(mode,seed) for mode in SHAPING_MODES for seed in GRID_SEEDS]
print(f"[defense] Training all {len(all_defense_runs)} policies from scratch.")
for run_number,(shaping_mode,seed) in enumerate(all_defense_runs,start=1):
    print(f"[defense {run_number}/{len(all_defense_runs)}] {shaping_mode=}, {seed=}")
    set_random_seed(seed)
    vec_env=make_training_vec_env_v2("full",seed,delay_rounds=BEST_DELAY_FOR_DEFENSE_TEST,
        n_envs=N_ENVS,shaping_mode=shaping_mode)
    model=PPO("MlpPolicy",vec_env,learning_rate=3e-4,n_steps=512,batch_size=256,
        n_epochs=10,gamma=0.999,gae_lambda=0.95,ent_coef=0.01,vf_coef=0.5,
        max_grad_norm=0.5,verbose=0,seed=seed,
        policy_kwargs=dict(net_arch=[64,64]),device=PPO_DEVICE)
    callback=LagrangeCallback(delta=EnvConfig().detection_limit)
    model.learn(total_timesteps=GRID_TIMESTEPS,callback=callback)
    cfg_shaped=EnvConfig(seed=0,obs_mode="full",delay_rounds=BEST_DELAY_FOR_DEFENSE_TEST,
                         shaping_mode=shaping_mode)
    df_eval=evaluate_rl_policy(cfg_shaped,model,vec_env,n_episodes=GRID_EVAL_EPISODES,
        seed0=TEST_SEED0+seed*GRID_EVAL_EPISODES)
    defense_rows.append(dict(shaping_mode=shaping_mode,seed=seed,
        mean_reward=float(df_eval["total_damage_reward"].mean()),
        frac_cusum_detected=float(df_eval["detected"].mean()),
        detection_rate=float(df_eval["detected"].mean()),
        metadata_overhead=float(df_eval["metadata_overhead"].mean()),
        throughput_loss=float(df_eval["metadata_throughput_loss"].mean()),
        latency=float(df_eval["metadata_latency"].mean()),
        feasible=bool(df_eval["detected"].mean()<=cfg_shaped.detection_limit),
        training_device=PPO_DEVICE))
    vec_env.close(); del model,vec_env; gc.collect()
defense_df=pd.DataFrame(defense_rows)
defense_df.to_csv(DEFENSE_CSV,index=False)
DEFENSE_STAGE_COMPLETE=(len(defense_df)==len(all_defense_runs))
print(defense_df.groupby("shaping_mode")[
    ["mean_reward","frac_cusum_detected","metadata_overhead","throughput_loss","latency"]].agg(["mean","std","count"]))


In [ ]:
# %% Cell 29 — Defense effectiveness plot
if ANALYSIS_READY and DEFENSE_STAGE_COMPLETE and os.path.exists(DEFENSE_CSV):
    defense_df = pd.read_csv(DEFENSE_CSV) if os.path.exists(DEFENSE_CSV) else pd.DataFrame()
    rl_grid_df = pd.read_csv(RL_RESULTS_CSV) if os.path.exists(RL_RESULTS_CSV) else pd.DataFrame()
    qber_pool_ref = rl_grid_df[(rl_grid_df.obs_mode == "qber_budget") &
                                (rl_grid_df.delay == BEST_DELAY_FOR_DEFENSE_TEST)]["mean_reward"].mean()

    fig, ax = plt.subplots(figsize=(7, 5))
    means = defense_df.groupby("shaping_mode")["mean_reward"].mean().reindex(SHAPING_MODES)
    stds = defense_df.groupby("shaping_mode")["mean_reward"].std().reindex(SHAPING_MODES)
    ax.bar(SHAPING_MODES, means, yerr=stds, capsize=4, color="tab:orange")
    ax.axhline(qber_pool_ref, color="tab:blue", ls="--", label="qber_pool reference (no messaging awareness)")
    ax.set_ylabel("mean availability-damage score (full agent)")
    ax.set_title(f"Does traffic shaping remove the advantage? (delay={BEST_DELAY_FOR_DEFENSE_TEST})")
    ax.legend()
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "defense_effectiveness.png"), dpi=130); plt.show()
else:
    print("[analysis] Defense plot will run when the defense sweep is complete.")


## Optional operating-regime and detector sweeps
These are explicit final-study runs. Leave the toggles off during a quick Kaggle smoke run; use disjoint test seeds and the full episode counts for reported results.


In [ ]:
# %% Optional sensitivity analyses — all run as part of the one-shot pipeline
CORE_RESULTS_READY = (GRID_STAGE_COMPLETE and BASELINES_STAGE_COMPLETE and
                      RISK_STAGE_COMPLETE and DEFENSE_STAGE_COMPLETE)
RUN_LOAD_RHO_SWEEP = (SESSION_STAGE == "sensitivity" or
                      (SESSION_STAGE == "all" and CORE_RESULTS_READY))
if RUN_LOAD_RHO_SWEEP:
    loads=(0.80,0.90,0.95,1.00,1.05)
    pool_sizes=(5000.0,10000.0,20000.0)
    rhos=(0.0,0.1,0.25,0.5,0.75,1.0)
    operating_rows=[]
    all_operating_points=[(load,pool,rho) for load in loads for pool in pool_sizes for rho in rhos]
    print(f"[load/rho] Running all {len(all_operating_points)} operating points.")
    for point_number,(load,pool_size,rho) in enumerate(all_operating_points,start=1):
        cfg_sweep=EnvConfig(load=load,rho=rho,pool_capacity_bits=pool_size,
                            obs_mode="full",delay_rounds=15)
        mean_emission,std_emission=fit_metadata_emissions(cfg_sweep)
        cfg_no_meta=copy.deepcopy(cfg_sweep); cfg_no_meta.rho=0.0
        factories={
            "HMMNoMetadata":lambda seed,c=cfg_no_meta,m=mean_emission,st=std_emission:
                MetadataHMMPolicy(c,m,st),
            "HMMMetadata":lambda seed,c=cfg_sweep,m=mean_emission,st=std_emission:
                MetadataHMMPolicy(c,m,st),
            "StateOracle":lambda seed,c=cfg_sweep: StateOraclePolicy(len(c.intercept_levels)-1),
        }
        episodes_per_point=min(GRID_EVAL_EPISODES,200)
        seed_point=TEST_SEED0+int(load*10000)+int(pool_size)+int(rho*1000)
        print(f"[load/rho {point_number}/{len(all_operating_points)}] load={load}, pool={pool_size}, rho={rho}")
        for agent,factory in factories.items():
            df_sweep=run_batch(cfg_sweep,factory,episodes_per_point,seed0=seed_point)
            operating_rows.append(dict(load=load,pool_size=pool_size,rho=rho,
                delay=cfg_sweep.delay_rounds,agent=agent,
                mean_reward=float(df_sweep.total_damage_reward.mean()),
                mean_damage=float(df_sweep.total_availability_damage.mean()),
                detection_rate=float(df_sweep.detected.mean()),
                mean_unmet_demand=float(df_sweep.total_exhaustion_deficit.mean()),
                n_episodes=episodes_per_point))
    operating_df=pd.DataFrame(operating_rows)
    operating_df.to_csv(os.path.join(RESULTS_DIR,"load_rho_sweep.csv"),index=False)
    voi=operating_df.pivot_table(index=["load","pool_size","rho","delay"],
                                 columns="agent",values="mean_reward").reset_index()
    voi["metadata_voi"]=voi["HMMMetadata"]-voi["HMMNoMetadata"]
    voi["metadata_voi_pct"]=np.where(np.abs(voi["HMMNoMetadata"])>1e-6,
        100.0*voi["metadata_voi"]/np.abs(voi["HMMNoMetadata"]),np.nan)
    voi.to_csv(os.path.join(RESULTS_DIR,"metadata_voi.csv"),index=False)
    display(voi)
    LOAD_RHO_COMPLETE=(len(operating_df)==len(all_operating_points)*len(factories))
else:
    LOAD_RHO_COMPLETE=False
    print("[load/rho] Not run because the primary study stages are incomplete.")

RUN_DETECTOR_PARAMETER_SWEEP = (SESSION_STAGE == "sensitivity" or
    (SESSION_STAGE == "all" and CORE_RESULTS_READY and LOAD_RHO_COMPLETE))
if RUN_DETECTOR_PARAMETER_SWEEP:
    detector_rows=[]
    detector_configs=[(25,0.003),(40,0.005),(50,0.007)]
    for sample_size,slack in detector_configs:
        cfg_det=EnvConfig(sample_size_qber=sample_size,cusum_slack=slack,
            cusum_threshold=None,target_false_alarm=0.01)
        cfg_det.cusum_threshold=calibrate_cusum_cached(cfg_det.qber_baseline,
            sample_size,cfg_det.max_rounds,slack,cfg_det.target_false_alarm)
        for f in cfg_det.intercept_levels:
            cfg_f=copy.deepcopy(cfg_det); cfg_f.intercept_levels=(f,)
            df_det=run_batch(cfg_f,lambda seed: ConstantActionPolicy(0),
                n_episodes=GRID_EVAL_EPISODES,seed0=TEST_SEED0)
            detector_rows.append(dict(sample_size=sample_size,slack=slack,f=f,
                detection_rate=float(df_det.detected.mean()),
                mean_detection_round=float(df_det.detection_round.mean())))
    detector_df=pd.DataFrame(detector_rows)
    detector_df.to_csv(os.path.join(RESULTS_DIR,"detector_parameter_sweep.csv"),index=False)
    display(detector_df)
    DETECTOR_COMPLETE=True
else:
    DETECTOR_COMPLETE=False
    print("[detector] Not run because the primary/load-rho stages are incomplete.")

RUN_RECURRENT_PPO = (SESSION_STAGE == "sensitivity" or
    (SESSION_STAGE == "all" and CORE_RESULTS_READY and LOAD_RHO_COMPLETE and DETECTOR_COMPLETE))
if RUN_RECURRENT_PPO:
    recurrent_cfg=EnvConfig(obs_mode="full",delay_rounds=50,detection_limit=0.05)
    recurrent_model,recurrent_vec_env,recurrent_summary,recurrent_test=train_recurrent_ppo(
        recurrent_cfg,total_timesteps=GRID_TIMESTEPS,test_episodes=GRID_EVAL_EPISODES)
    recurrent_base=os.path.join(MODEL_DIR,"recurrent_full_delay50")
    recurrent_model.save(recurrent_base)
    recurrent_vec_env.save(recurrent_base+"_vecnorm.pkl")
    with open(os.path.join(RESULTS_DIR,"recurrent_full_delay50_summary.json"),"w") as f:
        json.dump(recurrent_summary,f,indent=2)
    recurrent_test.to_csv(os.path.join(RESULTS_DIR,"recurrent_full_delay50_test.csv"),index=False)
    recurrent_vec_env.close()
    print(recurrent_summary)
    RECURRENT_COMPLETE=True
else:
    RECURRENT_COMPLETE=False
    print("[recurrent PPO] Not run because the preceding sensitivity stages are incomplete.")

if SESSION_STAGE == "all":
    pipeline_complete=(GRID_STAGE_COMPLETE and BASELINES_STAGE_COMPLETE and
        RISK_STAGE_COMPLETE and DEFENSE_STAGE_COMPLETE and LOAD_RHO_COMPLETE and
        DETECTOR_COMPLETE and RECURRENT_COMPLETE)
    if pipeline_complete:
        print("[pipeline complete] All experiment and sensitivity stages finished in this run.")
    else:
        print("[pipeline incomplete] A preceding stage failed before completing.")
